# Girardeau, Inema & Buzsaki (2017) on CRCNS hc-14

Paper: G. Girardeau, I. Inema, G. Buzsaki, "Reactivations of emotional memory in the hippocampus-amygdala
system during sleep", Nature Neuroscience 20:1634-1642 (2017), doi:10.1038/nn.4637.

Dataset: CRCNS hc-14, doi:10.6080/K0MS3QXD, read anonymously from the public bucket
`crcnsarchive` under `hc-14/live/`. CRCNS data sets are not versioned, so the version is recorded as
`unversioned`, and the MD5 sums of the files read, from the data set's `checksums.md5`, are printed
below so that a later change can be detected. The `live` folder is the only published copy (the `old`
folder holds a single superseded histology archive), so it is the one used; its data description is
version 0.6 of 28 November 2017, the original release.

Claims: `claims/crcns-hc-14/10.1038-nn.4637.yaml`, written from the version of record with its
supplement, and checked by a second reader.

What the data set holds, in short: the sorted spikes, positions and events of all 65 recorded sessions
of the 4 rats (61 of them are the paper's), and a set of undocumented summary tables computed by the
authors (cell types, firing rates per sleep state, ripple modulation, and the hippocampus-BLA pair
correlations in pre-NREM, run and post-NREM). The hand-scored sleep states are not in it, and the LFP is
deposited as 65 tar.gz archives of 3.0 to 9.0 GB (461 GB, printed in the survey) that can only be read
as a stream. So every claim on hippocampus-BLA reactivation, ripple modulation or state firing rates is
tested on the authors' tables. The pair table is checked against the raw spikes (units and run
correlations), StatesFR only for its whole-session rates and the bound of its wake column, and
PoissonRippleMod only for its internal consistency. Claims that need NREM
or REM for other pairs, for time within NREM or for the reactivation strength are tested on sleep states
decoded from the authors' per-state firing rates (StatesFR), an assumption of this attempt checked against
the authors' NREM and REM pair correlations.

Revision after the first verification: the states are decoded from StatesFR in place of the first
revision's immobility substitute, Rat08's structures are read from StructureMapping.ods, the air puff
location is also run corrected by the airpuffloc files, and the three REM claims are tested. After the
second verification only text was corrected, and outputs were added: the sizes of the LFP archives, the
run time, and the sessions where a 20-min window of the decay test has no pair defined.

| Claim | Tested | Data used |
|---|---|---|
| units-recorded | yes | raw clu files, structure lists |
| fig1b-dz-speed-anova, fig1b-dz-speed-posthoc, fig1b-speed-ratio, figs1b-ratio-each-rat, fig1c-slower-in-dz-post-run, figs1c-habituation, airpuff-location-no-bias | yes | raw positions and events |
| fig2c-bla-pyr-higher-in-sleep, fig2c-bla-interneurons, fig2c-skewed-rates | yes | StatesFR and SpikeParameters tables |
| fig3c-hpc-bla-ev-nrem, fig3c-hpc-bla-ev-rem-absent | yes | EVcontribs pair correlations |
| fig4-bla-ripple-modulation | yes | PoissonRippleMod table |
| figs5b-ripple-modulated-pairs-reactivate-more, run-correlated-pairs, fig5a-up-positive-pairs-change-most, figs6b-ev-nine-groups, figs9-ev-independent-of-rate | yes | EVcontribs pair correlations, raw spikes for the run |
| fig5b-top-contributors-gain-increase | yes | AllRippleCCGs table, EVcontribs |
| fig7b-strong-pairs-correlated-changes, fig7b-weak-pairs-not-correlated, fig7c-quartiles | yes | EVcontribs pair correlations, raw spikes of the test runs |
| bla-rates-safe-airpuff-absent | yes | raw spikes and positions |
| fig3c-decay, figs5a-hpc-bla-all-cells (windows), fig3d-pir-weaker-than-bla, figs4-intra-structure-nrem, ripple-rate-pre-post-absent, fig6a-airpuff-trajectory-reactivation, fig6a-safe-trajectory-no-reactivation, fig6b-airpuff-greater-than-safe, figs11a-whole-run-reactivation, figs11b-all-cells-both-trajectories, figs11c-all-cells-difference-absent | yes, on decoded NREM (assumption) | raw spikes, positions, ripples, StatesFR |
| fig3d-pir-reactivation, fig3e-cen-no-reactivation, figs4-intra-structure-rem-absent | yes, on decoded NREM and REM (assumption) | raw spikes, positions, StatesFR |

In [ ]:
import time, resource, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import hc14

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T0 = time.time()
print("dataset", hc14.DATASET, hc14.VERSION, "doi", hc14.DATASET_DOI)

In [ ]:
# Survey: the bucket's file list, the summary tables, and every session's clu archive (streamed).
keys = hc14.file_list()
clu_archives = sorted(k for k, s in keys if k.endswith("_clu.tar.gz"))
for suffix in ("_clu.tar.gz", ".lfp.tar.gz"):
    sz = np.array([s for k, s in keys if k.endswith(suffix)]) / 1e9
    print(f"archives *{suffix}: {len(sz)}, total {sz.sum():.1f} GB, each {sz.min():.2f} to {sz.max():.2f} GB")
print("files under hc-14/:", len(keys), "| session clu archives:", len(clu_archives))
summary = hc14.load_summary()
SESS = hc14.sessions(summary)
print("sessions in sessionindexing.mat:", len(SESS))
# Spikes are kept for the track subsessions (pre-run, run, post-run); on the hippocampal shanks, the
# shanks of the pair table's cells and the shanks labelled BLA, basolateral group, Pir or CeCM, the
# decoded NREM and REM periods of pre- and post-sleep too (and the first revision's substitute NREM).
_cells = summary["AllRats-EVcontribs"]["Index"].Cells.astype(int)
_hpc_sh = {rat: set(_cells[_cells[:, 0] == rat, 2]) for rat in (8, 10, 11)}
_lab = hc14.structure_shanks(summary)
extra = {f: {sh for (r, i, sh), L in _lab.items() if r == rat and i == idx and L in
             {"BLA", "BLV", "LaDL", "BMA", "BMP", "Pir", "CeCM"}} | _hpc_sh.get(rat, set())
            | set(_cells[(_cells[:, 0] == rat) & (_cells[:, 1] == idx)][:, [2, 4]].ravel()) for rat, idx, f in SESS}
# Rates of each unit in StatesFR columns 5 to 8 (the four states; see the data check), for the decoded
# sleep states (hc14.decode_states).
_sfr = summary["AllRats-StatesFR"]["StatesFR"]
rates = {f: {(int(r[2]), int(r[3])): r[5:9] for r in _sfr[(_sfr[:, 0] == rat) & (_sfr[:, 1] == idx)]}
         for rat, idx, f in SESS}
data = hc14.load_sessions([f for _, _, f in SESS], workers=3, extra_shanks=extra, state_rates=rates)
D = {f: d for (_, _, f), d in zip(SESS, data)}
del data
print(f"loaded in {time.time() - T0:.0f} s; peak memory of this process "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
md5 = hc14.md5sums()
read = ["AllRats-Variables.tar.gz", "Batch-Files.tar.gz"] + [f"Rat{r:02d}/Rat{r:02d}-Info.tar.gz" for r in (8, 9, 10, 11)] \
    + [f"{f[:5]}/{f}/{f}_clu.tar.gz" for _, _, f in SESS]
missing = [k for k in read if k not in md5]
print("MD5 sums of the files read (checksums.md5):")
for k in read:
    print(f"  {md5.get(k, 'not listed')}  {k}")
print("not listed:", missing)

## Data check

The summary tables carry no documentation (the data description says so). Their columns are
identified here against the raw files. Every check prints its numbers.

In [ ]:
ft = summary["AllRats-FinalType"]["finalType"].astype(int)          # rat, session, shank, clu, type
sfr = summary["AllRats-StatesFR"]["StatesFR"]                      # rat, session, shank, clu, index, 5 rates
spar = summary["SpikeParameters"]["SpikeParameters"]               # rat, session, shank, clu, ..., col 10
assert (sfr[:, :4] == ft[:, :4]).all() and (spar[:, :4] == ft[:, :4]).all()
print("units in the summary tables:", len(ft), "per rat:", dict(zip(*np.unique(ft[:, 0], return_counts=True))))
print("FinalType codes:", dict(zip(*np.unique(ft[:, 4], return_counts=True))))
print("SpikeParameters column 10 codes:", dict(zip(*np.unique(spar[:, 10], return_counts=True))))
print("cross table FinalType x SpikeParameters col 10:")
print(pd.crosstab(ft[:, 4], spar[:, 10]))

# Raw units: clusters >= 2 of every shank's clu file, per session.
rows = []
for rat, idx, f in SESS:
    d = D[f]
    raw = {(sh, c) for sh, c, n in d["units"]}
    m = (ft[:, 0] == rat) & (ft[:, 1] == idx)
    tab = {(int(a), int(b)) for a, b in ft[m, 2:4]}
    rows.append(dict(rat=rat, session=idx, folder=f, raw_units=len(raw), table_units=len(tab),
                     only_raw=len(raw - tab), only_table=len(tab - raw),
                     subsessions=" ".join(n for _, n, _, _ in d["subsessions"])))
units_df = pd.DataFrame(rows)
print(units_df.to_string())
print("raw units over the 61 sessions:", units_df.raw_units.sum(), "| table units:", units_df.table_units.sum())
print("sessions where raw and table differ:", units_df[(units_df.only_raw > 0) | (units_df.only_table > 0)].folder.tolist())

In [ ]:
# StatesFR: columns 5-8 are four state rates and column 9 the rate over the whole session.
# Column 9 against the raw spike count over the session duration:
ratio9, W, viol = [], [], []
for rat, idx, f in SESS:
    d = D[f]
    T = d["whl"].shape[0] / hc14.WHL_FS
    m = np.where((sfr[:, 0] == rat) & (sfr[:, 1] == idx))[0]
    n = {(sh, c): k for sh, c, k in d["units"]}
    raw_rate = np.array([n.get((int(a), int(b)), np.nan) for a, b in sfr[m, 2:4]]) / T
    ratio9.append(np.nanmedian(raw_rate / sfr[m, 9]))
    X, y = sfr[m, 5:9], sfr[m, 9]
    ok = ~np.isnan(X).any(1)
    w = np.linalg.lstsq(X[ok], y[ok], rcond=None)[0]       # time fraction of each state, if they partition the session
    W.append(w)
    track = hc14.intervals(d, "prerun") + hc14.intervals(d, "run") + hc14.intervals(d, "postrun")
    E = np.array([])
    ntrack = np.array([sum(np.sum((d["spikes"].get((int(a), int(b)), E) >= s0) & (d["spikes"].get((int(a), int(b)), E) < s1))
                           for s0, s1 in track) for a, b in sfr[m, 2:4]])
    if track:
        viol.append(np.mean(ntrack[:, None] > X * w * T * 1.05 + 3, axis=0))
W, viol = np.array(W), np.array(viol)
print("median over sessions of raw rate / column 9:", np.round(np.median(ratio9), 4))
print("fitted time fraction per state column (5, 6, 7, 8), median over sessions:", W.mean(0).round(3) * 0 + np.median(W, 0).round(3),
      "| sum:", np.median(W.sum(1)).round(3))
print("fraction of units whose spikes on the track exceed what the column allows (median, max over sessions):")
for j in range(4):
    print(f"  column {5 + j}: {np.median(viol[:, j]):.3f}, {viol[:, j].max():.3f}")

In [ ]:
# Which column is REM? Hippocampal interneurons fire more in REM than in NREM (theta); the
# hippocampal cells are those the EVcontribs pairs use.
ev = summary["AllRats-EVcontribs"]
cells = ev["Index"].Cells.astype(int)
key = lambda r: tuple(int(x) for x in r)
hpc_cells = {key(r) for r in cells[:, :4]}
bla_cells_ev = {key(r) for r in cells[:, [0, 1, 4, 5]]}
is_hpc = np.array([key(r) in hpc_cells for r in sfr[:, :4]])
for t, name in [(1, "pyramidal"), (2, "interneuron")]:
    m = is_hpc & (ft[:, 4] == t)
    X = sfr[m, 5:10]
    print(f"hippocampal {name}s (n = {m.sum()}): median rate per column 5..9", np.nanmedian(X, 0).round(3),
          "| median log2(column / column 6)", np.nanmedian(np.log2((X + .01) / (X[:, [1]] + .01)), 0).round(3))

Column 7 is the only one that can hold the spikes of the track epochs (at most 7.9% of units in a
session exceed it, against a majority for columns 5 and 8), and it takes 63% of the session: column 7
is wake. Columns 5 to 8 sum to the session (fitted fractions sum to 1). Column 6 takes 29% of the
session, column 5 about 5% and column 8 about 3%. Column 6 is read as NREM. For REM, hippocampal
interneurons fire more in column 5 than in column 6, as they do in REM against NREM, and less in column
8; column 5 is read as REM and column 8 as a fourth state (drowsy or intermediate). The REM column is an
assumption; the state firing rate claims are also run with column 8 as REM. The section "Decoded sleep
states" adds a check: epochs decoded as column 5 give hippocampus-BLA pair correlations that follow the
table's REM correlations (r about 0.8) and not its NREM ones (r about 0.09).

In [ ]:
COL = {"rem": 5, "nrem": 6, "wake": 7, "other": 8, "all": 9}
# PoissonRippleMod: rat, session, shank, clu, index, p (increase), p (decrease), signed surprise.
prm = summary["PoissonRippleMod"]["poissonripplemod"]
ok = ~np.isnan(prm[:, 5])
print("PoissonRippleMod rows:", len(prm), "rats:", dict(zip(*np.unique(prm[:, 0], return_counts=True))), "NaN rows:", (~ok).sum())
print("column 7 > 0 where column 5 < column 6:", np.mean((prm[ok, 7] > 0) == (prm[ok, 5] < prm[ok, 6])).round(4))
print("cells with both columns 5 and 6 below 0.0005:", np.sum((prm[ok, 5] < 5e-4) & (prm[ok, 6] < 5e-4)))
# EVcontribs.Allcorr: per hippocampus-BLA pair, columns (pre, run, post) for NREM ("sws"), REM, ripples.
A, Ap, Ty = ev["Allcorr"], ev["Allpval"], ev["Index"].Type
print("pairs:", len(cells), "| both pyramidal:", ((Ty == 1).all(1)).sum(),
      "| run column identical between sws and rem:", np.allclose(A.sws[:, 1], A.rem[:, 1], equal_nan=True))
pyr_ok = (Ty == 1).all(1) & ~np.isnan(A.sws).any(1)
print("pyramidal pairs defined in all three NREM columns:", pyr_ok.sum(),
      "| pooled EV, REV recomputed:", np.round(hc14.ev_rev(*A.sws[pyr_ok].T)[:2], 6),
      "| deposited:", round(ev["EV"].sws.pyr, 6), round(ev["REV"].sws.pyr, 6))

In [ ]:
# Run correlations of the pair table against the raw spikes: 50-ms bins over the run subsession(s),
# concatenated when there are several.
def session_pairs(rat, idx):
    m = np.where((cells[:, 0] == rat) & (cells[:, 1] == idx))[0]
    return m, [tuple(r) for r in cells[m][:, 2:4]], [tuple(r) for r in cells[m][:, 4:6]]

def pair_corr(d, m, h, b, ivs):
    """Correlation per pair over the bins of the intervals ivs (bins concatenated)."""
    uh, ub = sorted(set(h)), sorted(set(b))
    E = np.array([])
    Hc = np.hstack([hc14.bin_counts([d["spikes"].get(k, E) for k in uh], s0, s1) for s0, s1 in ivs])
    Bc = np.hstack([hc14.bin_counts([d["spikes"].get(k, E) for k in ub], s0, s1) for s0, s1 in ivs])
    R = hc14.cross_corr(Hc, Bc)
    ih, ib = {k: i for i, k in enumerate(uh)}, {k: i for i, k in enumerate(ub)}
    return np.array([R[ih[x], ib[y]] for x, y in zip(h, b)]), Hc.shape[1]

run_raw = np.full(len(cells), np.nan)
run_nbins = np.zeros(len(cells))
check = []
for rat, idx, f in SESS:
    m, h, b = session_pairs(rat, idx)
    if not len(m):
        continue
    d = D[f]
    run = hc14.intervals(d, "run")
    r, nb = pair_corr(d, m, h, b, run)
    run_raw[m], run_nbins[m] = r, nb
    ok = ~np.isnan(r) & ~np.isnan(A.sws[m, 1])
    check.append(dict(folder=f, pairs=len(m), compared=ok.sum(), r=np.corrcoef(r[ok], A.sws[m, 1][ok])[0, 1],
                      max_abs_diff=np.max(np.abs(r[ok] - A.sws[m, 1][ok]))))
check = pd.DataFrame(check)
print(check.round(5).to_string())
print("sessions:", len(check), "| pairs compared:", check.compared.sum(), "| worst r:", check.r.min().round(5))

In [ ]:
# The pre and post columns against correlations over the decoded NREM periods of pre-sleep and
# post-sleep (a check of the column order, and of the decoded NREM used for the claims that need
# sleep states; see the section "Decoded sleep states").
rows = []
for rat, idx, f in SESS:
    m, h, b = session_pairs(rat, idx)
    if not len(m):
        continue
    d = D[f]
    if not d["nrem"]["pre"] or not d["nrem"]["post"]:
        continue
    pre, _ = pair_corr(d, m, h, b, d["nrem"]["pre"])
    post, _ = pair_corr(d, m, h, b, d["nrem"]["post"])
    c = lambda x, y: np.corrcoef(x[~np.isnan(x) & ~np.isnan(y)], y[~np.isnan(x) & ~np.isnan(y)])[0, 1]
    rows.append(dict(folder=f, nrem_pre_min=sum(b_ - a_ for a_, b_ in d["nrem"]["pre"]) / 60,
                     nrem_post_min=sum(b_ - a_ for a_, b_ in d["nrem"]["post"]) / 60,
                     r_col0_pre=c(A.sws[m, 0], pre), r_col0_post=c(A.sws[m, 0], post),
                     r_col2_pre=c(A.sws[m, 2], pre), r_col2_post=c(A.sws[m, 2], post)))
sleepcheck = pd.DataFrame(rows)
print(sleepcheck.round(3).to_string())
print("medians:", sleepcheck.median(numeric_only=True).round(3).to_dict())

In [ ]:
# Clock check. Spikes, positions and events share one clock if (1) the hippocampal population rate
# follows running speed at zero lag on the track, and (2) speed drops right after the air puffs.
lags = np.arange(-40, 41, 4)     # samples of the .whl clock (25.6 ms)
rows = []
for rat, idx, f in SESS:
    d = D[f]
    run = hc14.intervals(d, "run")
    hp = [k for k in d["spikes"] if (rat, idx, k[0], k[1]) in hpc_cells]
    if not run or not hp:
        continue
    pos = hc14.position_cm(d["whl"])
    v = np.abs(hc14.linear_speed(pos[:, 0]))
    i0, i1 = int(run[0][0] * hc14.WHL_FS), int(run[-1][1] * hc14.WHL_FS)
    edges = np.arange(i0, i1 + 1) / hc14.WHL_FS
    pr = np.sum([np.histogram(d["spikes"][k], edges)[0] for k in hp], 0).astype(float)
    pr = np.convolve(pr, np.ones(8) / 8, mode="same")
    vv = v[i0:i1]
    cc = []
    for L in lags:
        a, b = (pr[L:], vv[:len(vv) - L]) if L >= 0 else (pr[:L], vv[-L:])
        ok = ~np.isnan(b)
        cc.append(np.corrcoef(a[ok], b[ok])[0, 1])
    rows.append(dict(folder=f, peak_lag_ms=lags[int(np.nanargmax(cc))] / hc14.WHL_FS * 1000, r_peak=np.nanmax(cc),
                     r_shift_10s=np.corrcoef(np.roll(pr, 391)[~np.isnan(vv)], vv[~np.isnan(vv)])[0, 1]))
clock = pd.DataFrame(rows)
print(clock.round(3).to_string())
print("sessions:", len(clock), "| median peak lag (ms):", clock.peak_lag_ms.median(), "| median r at peak:",
      clock.r_peak.median().round(3), "| median r with spikes shifted 10 s:", clock.r_shift_10s.median().round(3))
# Speed around the air puffs of the run, all sessions pooled.
win = np.arange(-78, 79)
trig = []
for rat, idx, f in SESS:
    d = D[f]
    if not len(d["puff"]):
        continue
    v = np.abs(hc14.linear_speed(hc14.position_cm(d["whl"])[:, 0]))
    for tp in d["puff"]:
        i = int(round(tp * hc14.WHL_FS))
        if 78 <= i < len(v) - 79:
            trig.append(v[i + win])
trig = np.array(trig)
mt = np.nanmean(trig, 0)
print("air puffs:", len(trig), "| mean speed 1 s before: %.1f cm/s, 0.5-1.5 s after: %.1f cm/s" %
      (np.nanmean(mt[(win >= -39) & (win < 0)]), np.nanmean(mt[(win >= 20) & (win < 59)])))

## units-recorded

> Over 61 sessions in 4 rats, 7,390 well-isolated units were recorded (2,444, 1,294, 1,138 and 2,514
> per rat), of which 2,038 in BLA, 782 in the central nuclei, 1,560 in piriform cortex and 1,210 in
> hippocampus.

Structures come from the deposited lists (Structures/*.mat for Rat09 and Rat10, Structures.txt for
Rat11, Batch-Files/Structures.batch for the BLA shanks of Rat08). Rat08's other structures, CeN and
piriform among them, are read from the text labels of the All sheet of Rat08-Info/StructureMapping.ods
(printed below), and the hippocampal shanks are taken as
the shanks of the hippocampal cells of the pair table (shanks 1-4 in Rat08, 17-20 in Rat10 and Rat11).

In [ ]:
lab = hc14.structure_shanks(summary)
hpc_shanks = {rat: set(cells[cells[:, 0] == rat, 2]) for rat in (8, 10, 11)}
B2 = {"BLA", "BLV", "LaDL", "BMA", "BMP"}
rows = []
for rat, idx, f in SESS:
    for sh, c, n in D[f]["units"]:
        L = lab.get((rat, idx, sh))
        rows.append(dict(rat=rat, session=idx, shank=sh, clu=c, label=L,
                         hpc=sh in hpc_shanks.get(rat, set())))
U = pd.DataFrame(rows)
print("sessions:", len(SESS), "(the bucket holds", len(clu_archives), "session archives; Rat11 sessions not in the",
      "paper:", sorted({k.split('/')[-2] for k in clu_archives} - {f for _, _, f in SESS}), ")")
print("units:", len(U), "per rat:", U.groupby("rat").size().to_dict())
print("units per label:", U.label.value_counts(dropna=False).to_dict())
res_units = {
    "BLA (B1, label BLA)": int((U.label == "BLA").sum()),
    "BLA (B2, basolateral group)": int(U.label.isin(B2).sum()),
    "central nuclei (CeCM)": int((U.label == "CeCM").sum()),
    "piriform (Pir)": int((U.label == "Pir").sum()),
    "hippocampus": int(U.hpc.sum()),
}
for k, v in res_units.items():
    print(f"  {k}: {v}   per rat {U[(U.label == 'BLA') if 'B1' in k else U.label.isin(B2) if 'B2' in k else (U.label == 'CeCM') if 'CeCM' in k else (U.label == 'Pir') if 'Pir' in k else U.hpc].groupby('rat').size().to_dict()}")
print("units in sessions with no structure list:", U[U.label.isna() & ~U.hpc].groupby("rat").size().to_dict())
# Rat08: the labels of StructureMapping.ods as written, units per label, and its BLA shanks against
# Structures.batch (which this attempt uses for BLA).
m08 = hc14.rat08_structure_mapping(summary["Rat08/StructureMapping.ods"], raw=True)
U8 = U[U.rat == 8]
print("Rat08 units per label as written in StructureMapping.ods:",
      pd.Series([m08.get((s, sh)) for s, sh in zip(U8.session, U8.shank)]).value_counts(dropna=False).to_dict())
b_bla = {(s, sh) for (r, s, sh), L in lab.items() if r == 8 and L == "BLA"}
o_bla = {k for k, v in m08.items() if v == "BLA"}
print(f"Rat08 BLA shank-days: Structures.batch {len(b_bla)}, StructureMapping.ods {len(o_bla)}, in both {len(b_bla & o_bla)}; "
      f"only in the batch file {sorted(b_bla - o_bla)}, only in the ods {sorted(o_bla - b_bla)}")
print("CeCM and Pir units per rat:", {L: U[U.label == L].groupby("rat").size().to_dict() for L in ("CeCM", "Pir")})
amb = U.label.isin(["VEn/Pir", "BLV/Pir"])
print(f"variant, Pir with the mixed labels VEn/Pir and BLV/Pir: {int((U.label == 'Pir').sum() + amb.sum())}")


**Result: reproduced.** The 61 sessions of sessionindexing.mat are the paper's (the bucket holds 65), the summary tables list 7,390 units with the paper's split by rat, and the raw clu files give 7,415, 25 more in Rat08-20130722. With the BLA label (B1), 2,038 units are in BLA, as in the paper; 783 in the central nuclei (paper 782) and 1,575 in piriform cortex (paper 1,560), with Rat08's labels read from the text of StructureMapping.ods; and 1,260 on the hippocampal shanks (paper 1,210). All are within the claims file's tolerance. Counting the mixed labels VEn/Pir and BLV/Pir as piriform gives 1,609. The ods and Structures.batch agree on Rat08's BLA shanks except one shank-day the ods adds.


## Behaviour: danger zone speeds

Positions are the LED midpoint in cm (0.43 cm per pixel); the track runs along x. The air puff location of
each session is the median position at the air puffs of the run (air puffs more than 40 cm from the
median are left out as stray), and the air puff direction the sign of the median velocity over the second
before the air puffs. The danger zone is the 20 cm before the location, on the approach in that
direction; the previous day's zone uses the previous session's location and direction. Speed per session
and epoch is the mean absolute speed over all samples in the zone, as the claims file has it; the mean
over samples moving in the air puff direction is a variant. A variant uses a corrected location: the
`airpuffloc` file's value in the 6 sessions that have one, and elsewhere the estimate moved back against
the direction of travel by the median offset between estimate and file (7.3 cm). The previous day's zone
is undefined after a session without a location.


In [ ]:
puff_rows = []
for rat, idx, f in SESS:
    d = D[f]
    run = hc14.intervals(d, "run")
    row = dict(rat=rat, session=idx, folder=f, loc=np.nan, dir=np.nan, n_puff=0, file=d["airpuffloc"])
    if run and len(d["puff"]):
        x = hc14.position_cm(d["whl"])[:, 0]
        v = hc14.linear_speed(x)
        pf = d["puff"][(d["puff"] >= run[0][0]) & (d["puff"] < run[-1][1])]
        ip = np.round(pf * hc14.WHL_FS).astype(int)
        px = x[ip]
        keep = np.abs(px - np.nanmedian(px)) < 40
        appr = np.array([np.nanmean(v[max(i - 39, 0):max(i - 4, 1)]) for i in ip])
        row.update(loc=np.nanmedian(px[keep]) if keep.any() else np.nan,
                   dir=np.sign(np.nanmedian(appr[keep])) if keep.any() else np.nan,
                   n_puff=len(pf), n_kept=int(keep.sum()), nan_at_puff=int(np.isnan(px).sum()))
    puff_rows.append(row)
P = pd.DataFrame(puff_rows)
f_rows = P[P.file.notna()]
off = []
for i, r in f_rows.iterrows():
    x_px, y_px, lab_dir = r.file.split()[:3]
    floc = float(x_px) * hc14.CM_PER_PX
    P.loc[i, "file_loc"] = floc
    off.append((r["loc"] - floc) * r["dir"])
    print(f"{r.folder}: airpuffloc file x = {floc:.1f} cm, {lab_dir}; from positions {r['loc']:.1f} cm, "
          f"direction {'LtoR' if r['dir'] > 0 else 'RtoL'}; estimate - file along the direction of travel {off[-1]:.1f} cm")
med_off = float(np.median(off))
# Corrected location (variant): the airpuffloc file where there is one, elsewhere the estimate moved back
# against the direction of travel by the median offset of the file sessions.
P["loc_corr"] = np.where(P.file_loc.notna(), P.file_loc, P["loc"] - P["dir"] * med_off)
print(f"median offset of the estimate along the direction of travel: {med_off:.1f} cm (n = {len(off)})")
# Previous day's zone: the previous session's location and direction; undefined after a session without them.
for c in ("loc", "dir", "loc_corr"):
    P["prev_" + c] = np.nan
for rat in P.rat.unique():
    ii = list(P.index[P.rat == rat])
    for last, i in zip(ii[:-1], ii[1:]):
        P.loc[i, ["prev_loc", "prev_dir", "prev_loc_corr"]] = P.loc[last, ["loc", "dir", "loc_corr"]].values
print(P.drop(columns="file").round(1).to_string())

In [ ]:
def dz_speeds(reading, loc="loc"):
    out = []
    for _, r in P.iterrows():
        d = D[r.folder]
        x = hc14.position_cm(d["whl"])[:, 0]
        v = hc14.linear_speed(x)
        t = np.arange(len(x)) / hc14.WHL_FS
        for ep in ("prerun", "run", "postrun"):
            iv = hc14.intervals(d, ep)
            if not iv:
                continue
            m = np.zeros(len(t), bool)
            for a, b in iv:
                m |= (t >= a) & (t < b)
            for zone, (L, Dr) in {"cur": (r[loc], r["dir"]), "prev": (r["prev_" + loc], r.prev_dir)}.items():
                if np.isnan(L):
                    continue
                inz = ((x >= L - hc14.DZ_CM) & (x <= L)) if Dr > 0 else ((x >= L) & (x <= L + hc14.DZ_CM))
                sel = m & inz & ~np.isnan(v)
                if reading == "dir":
                    sel &= np.sign(v) == Dr
                out.append(dict(folder=r.folder, rat=r.rat, epoch=ep, zone=zone,
                                speed=np.abs(v[sel]).mean() if sel.any() else np.nan))
    return pd.DataFrame(out).pivot_table(index=["folder", "rat"], columns=["epoch", "zone"], values="speed")

SP = {rd: dz_speeds(rd) for rd in ("all", "dir")}
# Variant: the corrected air puff location (file, or estimate moved back by the median offset).
SP.update({f"{rd}, corrected location": dz_speeds(rd, "loc_corr") for rd in ("all", "dir")})
for rd, w in SP.items():
    print(f"reading {rd}: sessions with any speed {len(w)}, with all six {len(w.dropna())}")
    print(w.median().round(1).to_string())

### fig1b-dz-speed-anova

> Speed in the danger zone depends on the epoch (pre-run, run, post-run) and on the zone (current
> against previous day's), with a significant interaction, in a two-way repeated measures ANOVA over 55
> sessions in 4 rats.

In [ ]:
anova_res = {}
for rd, w in SP.items():
    full = w.dropna()
    long = full.stack([0, 1], future_stack=True).rename("speed").reset_index()
    a = AnovaRM(long, "speed", "folder", within=["epoch", "zone"]).fit().anova_table
    anova_res[rd] = a
    print(f"reading {rd} ({len(full)} sessions, per rat {full.reset_index().groupby('rat').size().to_dict()})")
    print(a.round(5).to_string())
w = SP["all"]
miss = w[w.isna().any(axis=1)]
print(f"behaviour sessions: {P['loc'].notna().sum()} with an air puff location, of {len(P)}; with any speed {len(w)}, with all six {len(w.dropna())}")
print("sessions without all six speeds, and the speeds they lack:")
for (f_, rat), row in miss.iterrows():
    print(f"  {f_}: {[f'{a} {b}' for (a, b), v in row.items() if np.isnan(v)]}")
print("sessions with no air puff location:", P[P['loc'].isna() & (P.n_puff > 0)].folder.tolist(),
      "| with no run air puffs:", P[P.n_puff == 0].folder.tolist())


**Result: partially reproduced.** Over 50 sessions with all six speeds, with all samples in the zone (the claims file's reading), the epoch effect (F = 60.5) and the interaction (F = 16.4) have p < 0.0001, but the zone effect does not reach significance (F = 2.70, p = 0.11). With the samples moving in the air puff direction, zone p = 0.059. With the corrected air puff location, zone p = 0.49 (all samples) and 0.35 (direction-filtered), and the interaction stays significant (p = 0.0009 and p < 0.0001). The paper has 55 sessions; here 8 of 58 lack a speed: the first session of each rat (no previous zone), Rat09-20140329 (its previous session has no location), Rat09-20140328 (no tracked position at its air puffs), and Rat09-20140407 and Rat11-20150402 (no run air puffs, and epochs missing).


### fig1b-dz-speed-posthoc

> In pre-run rats run slower in the previous day's danger zone than in the current one, in run and
> post-run slower in the current danger zone than in the previous one, and the speed in the current
> danger zone is lower in post-run than in pre-run.

In [ ]:
def posthoc(w):
    out = []
    for ep in ("prerun", "run", "postrun"):
        a, b = w[(ep, "cur")], w[(ep, "prev")]
        ok = a.notna() & b.notna()
        t, p = stats.ttest_rel(a[ok], b[ok])
        out.append(dict(test=f"{ep}: current - previous", n=int(ok.sum()), t=t, p=p, mean_diff=(a[ok] - b[ok]).mean()))
    a, b = w[("prerun", "cur")], w[("postrun", "cur")]
    ok = a.notna() & b.notna()
    t, p = stats.ttest_rel(a[ok], b[ok])
    out.append(dict(test="current: pre-run - post-run", n=int(ok.sum()), t=t, p=p, mean_diff=(a[ok] - b[ok]).mean()))
    o = pd.DataFrame(out)
    o["significant at 0.05/4"] = o.p < 0.05 / 4
    return o

for rd, w in SP.items():
    print("reading", rd)
    print(posthoc(w).round(6).to_string())

**Result: partially reproduced.** Three of the four tests are significant at 0.05/4 with the stated directions: pre-run current faster than previous (t = 3.30, p = 0.0018), run current slower (t = -5.59, p = 1e-6), and current slower in post-run than in pre-run (t = 4.27, p = 8e-5). In post-run the current zone is slower than the previous one, but not significantly (t = -1.51, p = 0.14; p = 0.033 with the direction-filtered variant, still above 0.0125). With the corrected location and all samples, the pre-run test also fails (p = 0.041), so two of four hold, which would make the claim not reproduced under that reading.


### fig1b-speed-ratio

> The speed ratio (previous minus current over previous plus current danger zone speed) reverses
> between pre-run and post-run, with a significant epoch effect and every pair of epochs differing.

In [ ]:
def ratios(w):
    return pd.DataFrame({ep: (w[(ep, "prev")] - w[(ep, "cur")]) / (w[(ep, "prev")] + w[(ep, "cur")])
                         for ep in ("prerun", "run", "postrun")})

def ratio_tests(R):
    R = R.dropna()
    long = R.stack().rename("ratio").reset_index()
    long.columns = ["folder", "rat", "epoch", "ratio"]
    a = AnovaRM(long, "ratio", "folder", within=["epoch"]).fit().anova_table
    pairs = []
    for x, y in [("prerun", "run"), ("prerun", "postrun"), ("run", "postrun")]:
        t, p = stats.ttest_rel(R[x], R[y])
        pairs.append(dict(pair=f"{x} - {y}", t=t, p=p, sig_at_0_05_over_3=p < 0.05 / 3))
    return R, a, pd.DataFrame(pairs)

for rd, w in SP.items():
    R, a, pr = ratio_tests(ratios(w))
    print(f"reading {rd}: {len(R)} sessions; median ratio", R.median().round(3).to_dict())
    print(a.round(6).to_string()); print(pr.round(6).to_string())

**Result: reproduced (resting on the estimated air puff location).** The ratio's median is -0.088 in pre-run and 0.045 in post-run, the epoch effect is F = 25.4 (p < 0.0001, 50 sessions), and all three paired tests pass 0.05/3 in the stated directions (pre-run below run, t = -6.57; pre-run below post-run, t = -3.56, p = 0.0008; run above post-run, t = 3.97, p = 0.0002). With the corrected location and all samples, pre-run against post-run gives p = 0.031, above 0.05/3, which would make the claim partially reproduced; with the corrected location and direction-filtered samples all three pass.


### figs1b-ratio-each-rat

> The speed ratio changes across epochs in each of the four rats.

In [ ]:
for rd, w in SP.items():
    R = ratios(w).dropna()
    print("reading", rd)
    for rat, g in R.groupby(level="rat"):
        long = g.stack().rename("ratio").reset_index()
        long.columns = ["folder", "rat", "epoch", "ratio"]
        a = AnovaRM(long, "ratio", "folder", within=["epoch"]).fit().anova_table
        print(f"  rat {rat}: n = {len(g)}, F = {a['F Value'].iloc[0]:.2f}, p = {a['Pr > F'].iloc[0]:.3g}, "
              f"median pre-run {g.prerun.median():.3f}, post-run {g.postrun.median():.3f}")

**Result: reproduced (resting on the estimated air puff location).** The one-way ANOVA has p < 0.05 in each rat (p = 0.0012, 0.014, 4.6e-5, 0.026 for rats 8 to 11) and the median ratio is lower in pre-run than in post-run in all four. With the corrected location and all samples, rats 9 and 11 give p = 0.080 and 0.15 and rat 11's pre-run median is above its post-run median (not reproduced under that reading); direction-filtered, rat 11 gives p = 0.058 (partially reproduced).


### fig1c-slower-in-dz-post-run

> In the air puff direction rats slow down before the air puff location in run and post-run, while in
> pre-run speed rises smoothly along the track, so that speed in the current danger zone is lower in
> post-run than in pre-run.

Air-puff-centred speed curves as the criterion gives them: speed smoothed over 2 samples, 50 bins of the
normalized track (between the 1st and 99th percentiles of the run positions), laps in the air puff
direction, centred on the air puff bin and flipped so that the air puff is approached from the left; the
danger zone value is the mean of the 7 bins up to the air puff bin.

In [ ]:
NB = 50
def fig1c(loc):
    curves = {ep: [] for ep in ("prerun", "run", "postrun")}
    dzv = []
    for _, r in P.iterrows():
        if np.isnan(r[loc]):
            continue
        d = D[r.folder]
        x = hc14.position_cm(d["whl"])[:, 0]
        v = np.abs(hc14.linear_speed(x, smooth=2))
        run = hc14.intervals(d, "run")
        t = np.arange(len(x)) / hc14.WHL_FS
        mr = (t >= run[0][0]) & (t < run[-1][1])
        lo, hi = np.nanpercentile(x[mr], [1, 99])
        b = np.clip(((x - lo) / (hi - lo) * NB).astype(float), 0, NB - 1e-9)
        pb = int(np.clip((r[loc] - lo) / (hi - lo) * NB, 0, NB - 1))
        row = dict(folder=r.folder, rat=r.rat)
        for ep in curves:
            iv = hc14.intervals(d, ep)
            if not iv:
                continue
            s0, s1 = int(iv[0][0] * hc14.WHL_FS), int(iv[-1][1] * hc14.WHL_FS)
            prof = np.full(NB, np.nan)
            sums, cnt = np.zeros(NB), np.zeros(NB)
            for a, e, dr in hc14.laps(x[s0:s1]):
                if dr != r["dir"]:
                    continue
                seg = slice(s0 + a, s0 + e)
                bb = np.floor(b[seg]).astype(int)
                ok = ~np.isnan(v[seg]) & ~np.isnan(x[seg])
                np.add.at(sums, bb[ok], v[seg][ok]); np.add.at(cnt, bb[ok], 1)
            prof = np.where(cnt > 0, sums / np.maximum(cnt, 1), np.nan)
            # centre on the air puff bin, approached from the left
            rel = np.arange(NB) - pb
            if r["dir"] < 0:
                rel = -rel
            cen = np.full(2 * NB + 1, np.nan)
            cen[rel + NB] = prof
            curves[ep].append(cen)
            row[ep] = np.nanmean(cen[NB - 6:NB + 1])
        dzv.append(row)
    dzv = pd.DataFrame(dzv)
    ok = dzv.prerun.notna() & dzv.postrun.notna()
    rs = stats.ranksums(dzv.postrun[ok], dzv.prerun[ok])
    print(f"sessions: {ok.sum()}; median danger zone speed pre-run {dzv.prerun[ok].median():.1f}, run {dzv.run.median():.1f}, "
            f"post-run {dzv.postrun[ok].median():.1f} cm/s; rank-sum post vs pre z = {rs.statistic:.2f}, p = {rs.pvalue:.3g}")
    sr = stats.wilcoxon(dzv.postrun[ok], dzv.prerun[ok])
    print(f"variant, signed-rank paired by session: p = {sr.pvalue:.3g}")
    return curves, dzv
for loc in ("loc", "loc_corr"):
    print("air puff location:", "estimated" if loc == "loc" else "corrected")
    curves_, dzv_ = fig1c(loc)
    if loc == "loc":
        curves, dzv = curves_, dzv_
fig, ax = plt.subplots(figsize=(6, 3))
xb = np.arange(-NB, NB + 1)
for ep, c in zip(curves, ["0.6", "C3", "C0"]):
    mcur = np.nanmean(np.array(curves[ep]), 0)
    ax.plot(xb, mcur, color=c, label=ep)
    inr = (xb >= -30) & (xb <= 20)
    print(f"{ep}: mean curve range over bins -30..20: {np.nanmin(mcur[inr]):.1f} to {np.nanmax(mcur[inr]):.1f} cm/s")
ax.axvline(0, color="k", lw=0.5); ax.set_xlim(-30, 20); ax.set_xlabel("bins from air puff (approach from left)")
ax.set_ylabel("speed (cm/s)"); ax.legend(); plt.show()


**Result: reproduced (resting on the estimated air puff location).** In the air-puff-centred curves the danger zone speed is lower in post-run (median 41.0 cm/s) than in pre-run (62.4 cm/s) over 53 sessions, rank-sum p = 0.0013; with the corrected location, 41.8 against 57.2 cm/s, p = 0.0087.


### figs1c-habituation

> The aversive effect of the air puff gradually diminishes with training days.

Habituation index: laps divided by the run's duration in seconds, x 100, under three lap counts (all laps,
air puff direction laps, left-to-right laps). Training day is the session's rank among the rat's behaviour
sessions.

In [ ]:
hab = []
for _, r in P.iterrows():
    if np.isnan(r["loc"]):
        continue
    d = D[r.folder]
    run = hc14.intervals(d, "run")
    x = hc14.position_cm(d["whl"])[:, 0]
    s0, s1 = int(run[0][0] * hc14.WHL_FS), int(run[-1][1] * hc14.WHL_FS)
    L = hc14.laps(x[s0:s1])
    dur = sum(b - a for a, b in run)
    hab.append(dict(folder=r.folder, rat=r.rat, all=len(L) / dur * 100, airpuff=sum(l[2] == r["dir"] for l in L) / dur * 100,
                    ltor=sum(l[2] > 0 for l in L) / dur * 100))
hab = pd.DataFrame(hab)
hab["day"] = hab.groupby("rat").cumcount() + 1
for lc in ("all", "airpuff", "ltor"):
    rho, p = stats.spearmanr(hab.day, hab[lc])
    per = {rat: stats.spearmanr(g.day, g[lc]) for rat, g in hab.groupby("rat")}
    print(f"lap count {lc}: pooled rho = {rho:.3f}, p = {p:.3g} (n = {len(hab)}); per rat "
          + ", ".join(f"rat {k}: rho {v[0]:.2f} (p {v[1]:.2g})" for k, v in per.items())
          + f"; rats with rho > 0: {sum(v[0] > 0 for v in per.values())}")
print("index range (all laps):", hab["all"].min().round(3), "to", hab["all"].max().round(3))
# Laps per session and direction, and the run's track extent (1st to 99th percentile of x).
lapinfo = []
for _, r in P.iterrows():
    d = D[r.folder]
    run = hc14.intervals(d, "run")
    if not run:
        continue
    x = hc14.position_cm(d["whl"])[:, 0]
    seg = x[int(run[0][0] * hc14.WHL_FS):int(run[-1][1] * hc14.WHL_FS)]
    L = hc14.laps(seg)
    lo, hi = np.nanpercentile(seg, [1, 99])
    lapinfo.append(dict(folder=r.folder, laps_LtoR=sum(l[2] > 0 for l in L), laps_RtoL=sum(l[2] < 0 for l in L),
                        run_min=sum(b - a for a, b in run) / 60, x_p1=lo, x_p99=hi, extent_cm=hi - lo))
lapinfo = pd.DataFrame(lapinfo)
print(lapinfo.round(1).to_string())
print("track extent (cm), min/median/max:", lapinfo.extent_cm.min().round(1), lapinfo.extent_cm.median().round(1),
      lapinfo.extent_cm.max().round(1), "| sessions with fewer than 5 laps in a direction:",
      lapinfo[(lapinfo.laps_LtoR < 5) | (lapinfo.laps_RtoL < 5)].folder.tolist())


**Result: reproduced.** The habituation index rises with training day under every lap count (pooled rho = 0.69 to 0.70, p < 1e-8, 55 sessions), and rho > 0 in all four rats. The track extent over the run is 190 to 211 cm in every session; Rat08-20130708, -0709, -0711 and Rat09-20140328 have fewer than 5 laps in a direction.


### airpuff-location-no-bias

> The location of the air puff on the track does not correlate with the speed in the current danger zone
> in pre-run, run or post-run.

In [ ]:
for rd, w in SP.items():
    lc = "loc_corr" if "corrected" in rd else "loc"
    print("reading", rd)
    for ep in ("prerun", "run", "postrun"):
        s = w[(ep, "cur")].dropna()
        rows = P.set_index("folder").loc[s.index.get_level_values(0)]
        lo, hi = 0, None
        # air puff position normalized to the track (run positions, 1st-99th percentiles), flipped for right-to-left
        norm = []
        for f_, rr in rows.iterrows():
            d = D[f_]
            x = hc14.position_cm(d["whl"])[:, 0]
            run = hc14.intervals(d, "run")
            seg = x[int(run[0][0] * hc14.WHL_FS):int(run[-1][1] * hc14.WHL_FS)]
            a, b = np.nanpercentile(seg, [1, 99])
            z = (rr[lc] - a) / (b - a)
            norm.append(z if rr["dir"] > 0 else 1 - z)
        r_, p_ = stats.pearsonr(norm, s.values)
        print(f"  {ep}: n = {len(s)}, r = {r_:.3f}, p = {p_:.3g}")

**Result: not reproduced (resting on the estimated air puff location).** The normalized air puff position correlates with the current danger zone speed in pre-run (r = -0.40, p = 0.0028) and in run (r = -0.34, p = 0.012), though not in post-run (r = -0.09, p = 0.50). With direction-filtered speeds, run is still significant (r = -0.30, p = 0.026). With the corrected location, all samples give pre-run p = 0.015 and run p = 0.037; direction-filtered, none is significant (p = 0.35, 0.064, 0.51), so the outcome depends on the location and on the speed reading.


## Firing rates per state (Figure 2c)

Rates per state are the StatesFR table's columns: wake column 7, NREM column 6, REM column 5 (the
assumption stated in the data check; column 8 as REM is shown as a variant). Monosynaptically identified
cells are SpikeParameters column 10 (1 excitatory, 2 inhibitory), the only code that splits the final
types into a connected and an unconnected part. BLA units are those on shanks labelled BLA (B1) or in the
basolateral group (B2); Rat08's sessions 10 to 13 have no BLA shanks listed.

In [ ]:
lab_unit = np.array([lab.get((int(r), int(s), int(sh))) for r, s, sh in ft[:, :3]], dtype=object)
in_bla = {"B1": lab_unit == "BLA", "B2": np.isin(lab_unit.astype(str), list(B2))}
mono = spar[:, 10]
def ratio(state, rem_col=5):
    c = rem_col if state == "rem" else COL[state]
    s, w = sfr[:, c], sfr[:, COL["wake"]]
    return (s - w) / (s + w)
for rd, m in in_bla.items():
    print(f"{rd}: BLA units {m.sum()}; monosynaptic pyramidal {np.sum(m & (mono == 1))}, monosynaptic interneurons "
          f"{np.sum(m & (mono == 2))}, others {np.sum(m & (mono == 0))}; per rat {pd.Series(ft[m, 0]).value_counts().sort_index().to_dict()}")
print("BLA cells of the pair table (3 rats):", len(bla_cells_ev))

### fig2c-bla-pyr-higher-in-sleep

> Monosynaptically identified BLA pyramidal cells fire more during REM sleep and, less so, during NREM
> sleep than during wakefulness.

In [ ]:
def state_test(sel, label):
    out = []
    for rem_col in (5, 8):
        for st in ("nrem", "rem"):
            if st == "nrem" and rem_col == 8:
                continue
            r = ratio(st, rem_col)[sel]
            r = r[~np.isnan(r)]
            w = stats.wilcoxon(r)
            out.append(dict(cells=label, ratio=f"{st}/wake" + (" (REM = column 8)" if rem_col == 8 else ""),
                            n=len(r), median=np.median(r), frac_pos=np.mean(r > 0), p=w.pvalue))
    return pd.DataFrame(out)
for rd, m in in_bla.items():
    print(state_test(m & (mono == 1), f"{rd} mono pyramidal").round(6).to_string())
    print(state_test(m & (ft[:, 4] == 1), f"{rd} all final-type pyramidal").round(6).to_string())
    for rat in (8, 9, 10, 11):
        sel = m & (mono == 1) & (ft[:, 0] == rat)
        rn, rr = ratio("nrem")[sel], ratio("rem")[sel]
        print(f"  {rd} rat {rat}: n = {sel.sum()}, median NREM/wake {np.nanmedian(rn):.3f}, REM/wake {np.nanmedian(rr):.3f}")

**Result: reproduced (resting on the REM column assumption).** The BLA label gives exactly the paper's 675 monosynaptic pyramidal cells, 175 interneurons and 1,188 others. Their NREM/wake ratios (median 0.26) and REM/wake ratios (median 0.36) are above 0 (p < 1e-15 each), and the REM ratio is the larger. With column 8 as REM, the REM ratio is 0.085, above 0 but smaller than the NREM ratio, which would make the claim partially reproduced. The decoded-state check below supports column 5 as REM.


### fig2c-bla-interneurons

> Monosynaptically identified BLA interneurons do not change their firing rate between REM sleep and
> wakefulness, and slightly decrease it during NREM sleep compared to wakefulness.

In [ ]:
for rd, m in in_bla.items():
    print(state_test(m & (mono == 2), f"{rd} mono interneurons").round(6).to_string())
    for rat in (8, 9, 10, 11):
        sel = m & (mono == 2) & (ft[:, 0] == rat)
        print(f"  {rd} rat {rat}: n = {sel.sum()}, median NREM/wake {np.nanmedian(ratio('nrem')[sel]):.3f}, "
              f"REM/wake {np.nanmedian(ratio('rem')[sel]):.3f}")

**Result: reproduced (resting on the REM column assumption).** For the 175 monosynaptic BLA interneurons, the REM/wake ratio does not differ from 0 (median 0.020, p = 0.21), and the NREM/wake ratio is below 0 (median -0.071, p < 1e-6). With column 8 as REM, the REM/wake ratio would be below 0 (p < 1e-6) and the REM part would fail. The decoded-state check below supports column 5 as REM.


### fig2c-skewed-rates

> The firing rates of BLA pyramidal cells follow a skewed distribution in NREM sleep, REM sleep and
> wakefulness.

In [ ]:
for rd, m in in_bla.items():
    sel = m & (mono == 1)
    for st in ("nrem", "rem", "wake"):
        r = sfr[sel, COL[st]]
        r = r[~np.isnan(r)]
        pos = r[r > 0]
        print(f"{rd} {st}: n = {len(r)} ({np.sum(r == 0)} zero rates left out of the log), skewness of rates "
              f"{stats.skew(r):.2f}, of log10 rates {stats.skew(np.log10(pos)):.2f}; range {r.min():.3f} to {r.max():.2f} Hz")
    r8 = sfr[sel, 8]; r8 = r8[~np.isnan(r8)]
    print(f"{rd} column 8 (REM variant): skewness {stats.skew(r8):.2f}, of log10 {stats.skew(np.log10(r8[r8 > 0])):.2f}")

**Result: reproduced.** In NREM, REM and wake the rates of the 675 cells have positive skewness (3.39, 14.03, 7.35), and the log rates are much less skewed (-0.41, -0.58, 0.12). Column 8 as REM gives the same (7.67 and -0.02).

## Hippocampus-BLA reactivation (EVcontribs pair correlations)

Per session, EV and REV from the pre-NREM, run and post-NREM correlations of the pair table (the formula
of the claims file, which reproduces the deposited pooled EV exactly). Sessions need at least 6 cells of
the type used in each structure, counted over the session's pairs. The 180-s REM requirement cannot be
checked from the table (no state durations are deposited per session); REM sessions are those with
defined REM correlations, and the requirement is applied with decoded REM durations as a variant.

In [ ]:
def session_ev(kind="sws", pyr=True, pair_mask=None, min_cells=6):
    C = getattr(A, kind)
    rows = []
    for rat, idx, f in SESS:
        m = (cells[:, 0] == rat) & (cells[:, 1] == idx)
        if pyr:
            m &= (Ty == 1).all(1)
        if pair_mask is not None:
            m &= pair_mask
        m = np.where(m)[0]
        if not len(m):
            continue
        nh = len({tuple(r) for r in cells[m][:, 2:4]})
        nb = len({tuple(r) for r in cells[m][:, 4:6]})
        ev_, rev_, n = hc14.ev_rev(*C[m].T)
        rows.append(dict(rat=rat, session=idx, folder=f, n_hpc=nh, n_bla=nb, pairs=n, EV=ev_, REV=rev_))
    df = pd.DataFrame(rows)
    df = df[(df.n_hpc >= min_cells) & (df.n_bla >= min_cells) & df.EV.notna()]
    df["diff"] = (df.EV - df.REV) * 100
    return df

def ev_test(df, label):
    w = stats.wilcoxon(df.EV, df.REV)
    print(f"{label}: sessions {len(df)} (per rat {df.groupby('rat').size().to_dict()}), median EV {df.EV.median()*100:.2f}%, "
          f"REV {df.REV.median()*100:.2f}%, mean EV-REV {df['diff'].mean():.2f} +- {df['diff'].sem():.2f}% (s.e.m.), "
          f"EV > REV in {np.sum(df.EV > df.REV)}, signed-rank p = {w.pvalue:.3g}")
    return w.pvalue

### fig3c-hpc-bla-ev-nrem

> Hippocampus-BLA pyramidal cell pairs reactivate during post-training NREM sleep, with EV larger than
> REV across sessions.

In [ ]:
ev_nrem_pyr = session_ev("sws", True)
p_nrem = ev_test(ev_nrem_pyr, "NREM, pyramidal pairs")
for rat, g in ev_nrem_pyr.groupby("rat"):
    print(f"  rat {rat}: {len(g)} sessions, mean EV-REV {g['diff'].mean():.2f}%, EV > REV in {np.sum(g.EV > g.REV)}")
for k in (4, 8):
    ev_test(session_ev("sws", True, min_cells=k), f"  variant, at least {k} cells")

**Result: reproduced.** Over 25 sessions (the paper's number; 9, 9 and 7 per rat), EV exceeds REV in 22, mean EV - REV 3.14 +- 0.73% (the paper's 3.13 +- 0.73%), signed-rank p = 4.5e-5. This rests on the authors' NREM pair correlations, whose run column matches the raw spikes.

### fig3c-hpc-bla-ev-rem-absent

> Hippocampus-BLA pyramidal pairs show no significant reactivation during REM sleep.

In [ ]:
ev_rem_pyr = session_ev("rem", True)
p_rem = ev_test(ev_rem_pyr, "REM, pyramidal pairs")
print("median EV - REV:", ev_rem_pyr["diff"].median().round(3), "%")# Variant: the 180-s REM requirement, with REM durations from the decoded states (section "Decoded sleep states").
rem_ok = {(r, i) for r, i, f in SESS if all(sum(b - a for a, b in D[f]["rem"][s]) > 180 for s in ("pre", "post"))}
_ = ev_test(ev_rem_pyr[[(a, b) in rem_ok for a, b in zip(ev_rem_pyr.rat, ev_rem_pyr.session)]],
        "variant, sessions with more than 180 s of decoded REM in pre- and post-sleep")


**Result: reproduced.** In REM, EV and REV do not differ over 25 sessions (mean EV - REV 0.06%, p = 0.73). Keeping the 20 sessions with more than 180 s of decoded REM in pre- and post-sleep (the paper has 23) gives p = 0.55.


### fig4-bla-ripple-modulation

> A fraction of BLA neurons are positively modulated during hippocampal SPW-Rs (42 of 163 interneurons,
> 25.8%; 137 of 1,233 pyramidal cells, 11.1%) or negatively modulated (24 of 163 interneurons, 14.7%; 102 of
> 1,233 pyramidal cells, 8.3%).

PoissonRippleMod columns 5 and 6 are the p-values of an increase and of a decrease. BLA cells are taken
three ways: the B1 and B2 shank labels, and the BLA cells of the pair table; cell types are final types.

In [ ]:
prm_key = {key(r[:4]): r for r in prm}
rip_sessions = {(r, s) for r, s in cells[:, :2]}
def ripmod(sel_keys, thr):
    out = {}
    for t, name in [(1, "pyramidal"), (2, "interneuron")]:
        ks = [k for k in sel_keys if k in ft_type and ft_type[k] == t and k in prm_key and not np.isnan(prm_key[k][5])]
        up = sum(prm_key[k][5] < thr for k in ks); dn = sum(prm_key[k][6] < thr for k in ks)
        out[name] = (len(ks), up, dn)
    return out
ft_type = {key(r[:4]): int(r[4]) for r in ft}
sets = {
    "B1 (BLA label)": [key(r[:4]) for r, m in zip(ft, in_bla["B1"]) if m and (r[0], r[1]) in rip_sessions],
    "B2 (basolateral group)": [key(r[:4]) for r, m in zip(ft, in_bla["B2"]) if m and (r[0], r[1]) in rip_sessions],
    "pair-table BLA cells": sorted(bla_cells_ev),
}
ripmod_res = {}
for name, ks in sets.items():
    for thr in (0.0005, 0.001):
        o = ripmod(ks, thr)
        ripmod_res[(name, thr)] = o
        (npy, upp, dnp), (ni, upi, dni) = o["pyramidal"], o["interneuron"]
        print(f"{name}, p < {thr} per direction: interneurons up {upi}/{ni} ({100*upi/ni:.1f}%), down {dni}/{ni} "
              f"({100*dni/ni:.1f}%); pyramidal up {upp}/{npy} ({100*upp/npy:.1f}%), down {dnp}/{npy} ({100*dnp/npy:.1f}%)")

**Result: reproduced.** With the BLA label and the code's threshold, 42 of 163 interneurons (25.8%) and 137 of 1,232 pyramidal cells (11.1%) are upmodulated, and 24 (14.7%) and 102 (8.3%) downmodulated, the paper's numbers (one pyramidal cell has no p-value). At p < 0.001 per direction the pyramidal percentages are 12.2% and 9.9%.

## Pair groups: ripple modulation and run correlation

The BLA partner of each pair is upmodulated or downmodulated when its PoissonRippleMod p-value of an
increase or of a decrease is below 0.0005 (the claims file's threshold from the code), and the pair's run
correlation is positive or negative when its p-value (the table's `Allpval`, the Matlab `corr` p-value)
is below 0.01.

In [ ]:
bla_key = [key(r) for r in cells[:, [0, 1, 4, 5]]]
p_up = np.array([prm_key[k][5] if k in prm_key else np.nan for k in bla_key])
p_dn = np.array([prm_key[k][6] if k in prm_key else np.nan for k in bla_key])
mod = np.where(p_up < 5e-4, "up", np.where(p_dn < 5e-4, "down", "none"))
mod[np.isnan(p_up)] = "nan"
r_run, p_run = A.sws[:, 1], Ap.sws[:, 1]
runc = np.where((p_run < 0.01) & (r_run > 0), "pos", np.where((p_run < 0.01) & (r_run < 0), "neg", "ns"))
print("pair table's own RippleMod code against this classification:")
print(pd.crosstab(ev["Index"].RippleMod, mod))
# p-values of the table against the t-distribution p of the raw run correlation
pr = hc14.corr_p(run_raw, run_nbins)
ok = ~np.isnan(pr) & ~np.isnan(p_run)
print("run p-values, raw against table: fraction agreeing on p < 0.01:", np.mean((pr[ok] < 0.01) == (p_run[ok] < 0.01)).round(4))

### run-correlated-pairs

> During run, 2,521 of 37,660 hippocampus-BLA pyramidal pairs (6.69%) are significantly positively
> correlated, 1,258 (3.34%) negatively correlated and 33,881 (89.96%) not reliably correlated; the pairs
> come from rat 1 (16,056), rat 3 (3,836) and rat 4 (17,768).

In [ ]:
PY = pyr_ok
n = PY.sum()
for src, rr, pp in [("table", r_run, p_run), ("raw spikes", run_raw, pr)]:
    pos = np.sum(PY & (pp < 0.01) & (rr > 0)); neg = np.sum(PY & (pp < 0.01) & (rr < 0))
    print(f"{src}: pairs {n}, positive {pos} ({100*pos/n:.2f}%), negative {neg} ({100*neg/n:.2f}%), "
          f"not significant {n-pos-neg} ({100*(n-pos-neg)/n:.2f}%)")
print("pairs per rat:", pd.Series(cells[PY, 0]).value_counts().sort_index().to_dict())
tab = pd.crosstab(pd.Series(mod[PY], name="BLA cell"), pd.Series(runc[PY], name="run correlation"))
print(tab)
print("group 1 (upmodulated, positive):", tab.loc["up", "pos"])

**Result: reproduced.** Of 37,660 pyramidal pairs (16,056, 3,836 and 17,768 per rat), 2,521 (6.69%) are positively and 1,258 (3.34%) negatively correlated in run, and 595 pairs have an upmodulated BLA cell and a positive correlation, all the paper's numbers. From the raw spikes, 2,521 (6.69%) and 1,259 (3.34%).

### fig5a-up-positive-pairs-change-most

> Hippocampus-BLA pairs with a significantly positive run correlation and an upmodulated BLA partner
> show the largest increase of correlation from pre-NREM to post-NREM of the nine groups, and are the
> only group that differs from all others.

In [ ]:
def nine_groups(mask, label):
    ch = A.sws[:, 2] - A.sws[:, 0]
    g = pd.DataFrame({"group": [f"{a}/{b}" for a, b in zip(mod, runc)], "change": ch})[mask & (mod != "nan")]
    f_, p_ = stats.f_oneway(*[x.change.values for _, x in g.groupby("group")])
    means = g.groupby("group").change.agg(["mean", "count"]).sort_values("mean", ascending=False)
    print(f"{label}: one-way ANOVA F = {f_:.2f}, p = {p_:.3g}, pairs {len(g)}")
    print(means.round(5).to_string())
    tk = pairwise_tukeyhsd(g.change, g.group)
    t = pd.DataFrame(tk.summary().data[1:], columns=tk.summary().data[0])
    t1 = t[(t.group1 == "up/pos") | (t.group2 == "up/pos")]
    print("Tukey-Kramer, group up/pos against each other group:")
    print(t1[["group1", "group2", "meandiff", "p-adj"]].to_string())
    g1 = g[g.group == "up/pos"].change
    bon = []
    for name, x in g.groupby("group"):
        if name == "up/pos":
            continue
        tt = stats.ttest_ind(g1, x.change)
        bon.append(dict(other=name, p=tt.pvalue, p_bonferroni=min(1, tt.pvalue * 8)))
    bon = pd.DataFrame(bon)
    print("Bonferroni (8 t-tests against group up/pos):"); print(bon.round(6).to_string())
    return means, t1, bon
fig5a_res = nine_groups(PY, "pyramidal pairs")
fig5a_all = nine_groups(~np.isnan(A.sws).any(1), "variant, all cell pairs")

**Result: reproduced.** The nine groups differ (F = 29.08, p = 1.1e-45, the paper's values), the upmodulated, positively correlated group has the largest mean change (0.0039), and it differs from each other group at p < 0.01 by Tukey-Kramer (all p-adj < 0.001) and by Bonferroni.

### figs6b-ev-nine-groups

> EV computed within each of the nine pair groups confirms that pairs with positive run correlation and
> an upmodulated BLA cell contribute most to reactivations.

In [ ]:
rows = []
for a in ("up", "none", "down"):
    for b in ("pos", "ns", "neg"):
        m = PY & (mod == a) & (runc == b)
        e, r_, k = hc14.ev_rev(*A.sws[m].T)
        rows.append(dict(group=f"{a}/{b}", pairs=k, EV=e * 100, REV=r_ * 100, diff=(e - r_) * 100))
g9 = pd.DataFrame(rows).sort_values("diff", ascending=False)
print(g9.round(3).to_string())

**Result: reproduced.** The upmodulated, positively correlated group has the largest EV - REV of the nine (14.7%, next 2.4%).

### figs5b-ripple-modulated-pairs-reactivate-more

> Reactivations are larger for pairs with a ripple-modulated BLA cell than for pairs with a non-modulated
> BLA cell (mean EV - REV 7.16% against 1.90% for pyramidal pairs, 6.07% against 1.76% for all pairs).

In [ ]:
for pyr, label in [(True, "pyramidal pairs"), (False, "all pairs")]:
    modd = session_ev("sws", pyr, pair_mask=np.isin(mod, ["up", "down"]))
    non = session_ev("sws", pyr, pair_mask=(mod == "none"))
    pm = ev_test(modd, f"{label}, ripple-modulated BLA cell")
    pn = ev_test(non, f"{label}, non-modulated BLA cell")
    rs = stats.ranksums(modd["diff"], non["diff"])
    print(f"  rank-sum modulated vs non-modulated EV-REV: z = {rs.statistic:.2f}, p = {rs.pvalue:.3g}; "
          f"means {modd['diff'].mean():.2f} vs {non['diff'].mean():.2f}%")

**Result: reproduced.** EV - REV is larger for pairs with a ripple-modulated BLA cell, for pyramidal pairs (7.16 against 1.67%, rank-sum p = 0.0040) and for all pairs (6.08 against 1.81%, p = 0.0048); all four means are within the paper's s.e.m. The non-modulated sets have 22 and 23 sessions, where the paper has 18 and 19.

### figs9-ev-independent-of-rate

> EV does not depend on firing rates when pairs are pooled by the BLA cell's rate, the hippocampal cell's
> rate or their combined rate.

Readings for the rate: the whole-session rate (StatesFR column 9, which is also the pair table's `FR`
columns) and the run rate from the raw spikes.

In [ ]:
whole = {key(r[:4]): r[9] for r in sfr}
_name = {(r, i): f for r, i, f in SESS}
def run_rate(k):
    rat, idx, sh, c = k
    d = D[_name[(rat, idx)]]
    run = hc14.intervals(d, "run")
    t = d["spikes"].get((sh, c), np.array([]))
    return sum(np.sum((t >= a) & (t < b)) for a, b in run) / sum(b - a for a, b in run)
hk = [key(r) for r in cells[:, :4]]
rates = {
    "whole session": (np.array([whole[k] for k in hk]), np.array([whole[k] for k in bla_key])),
    "run": (np.array([run_rate(k) for k in hk]), np.array([run_rate(k) for k in bla_key])),
}
print("pair table FR columns equal to StatesFR column 9:",
      np.allclose(ev["Index"].FR[:, 0], rates["whole session"][0]), np.allclose(ev["Index"].FR[:, 1], rates["whole session"][1]))
for rname, (rh, rb) in rates.items():
    for gname, val in [("mean of both", (rh + rb) / 2), ("BLA cell", rb), ("hippocampal cell", rh)]:
        v = val[PY]
        q = np.quantile(v, np.linspace(0, 1, 9))
        oc = np.clip(np.searchsorted(q, v, side="right") - 1, 0, 7)
        dd, nev = [], 0
        idx_py = np.where(PY)[0]
        for o in range(8):
            mm = idx_py[oc == o]
            e, r_, k = hc14.ev_rev(*A.sws[mm].T)
            dd.append((e - r_) * 100); nev += e > r_
        rho, p = stats.spearmanr(np.arange(8), dd)
        print(f"{rname} rate, {gname}: EV > REV in {nev}/8 octiles; EV-REV per octile (%) {np.round(dd, 2)}; "
              f"Spearman with octile rho = {rho:.2f}, p = {p:.3g}")

**Result: reproduced, under the whole-session rate reading.** With whole-session rates (the pair table's own FR), EV exceeds REV in 8 of 8 octiles for every grouping and no Spearman trend is significant (p = 0.35, 0.086, 0.74). With run rates, EV - REV rises with the rate (rho = 1.00 and 0.98 for the mean and BLA rates).

## Contributions of pairs to reactivation (Figure 7)

Contribution of each pyramidal pair: EV over all 37,660 pooled pairs minus EV without the pair (Online
Methods). It is computed here in closed form from running sums and checked against the table's
`EV.minus` (EV without each pair).

In [ ]:
def loo_ev(x, y, z):
    """EV of (pre x, run y, post z) with each element left out in turn (vectorized)."""
    n = len(x) - 1
    S = {k: v.sum() for k, v in dict(x=x, y=y, z=z, xx=x*x, yy=y*y, zz=z*z, xy=x*y, xz=x*z, yz=y*z).items()}
    sx, sy, sz = S["x"] - x, S["y"] - y, S["z"] - z
    def corr(sab, sa, sb, saa, sbb):
        cov = sab - sa * sb / n
        return cov / np.sqrt((saa - sa**2 / n) * (sbb - sb**2 / n))
    r_rp = corr(S["xy"] - x*y, sy, sx, S["yy"] - y*y, S["xx"] - x*x)
    r_ro = corr(S["yz"] - y*z, sy, sz, S["yy"] - y*y, S["zz"] - z*z)
    r_po = corr(S["xz"] - x*z, sx, sz, S["xx"] - x*x, S["zz"] - z*z)
    return ((r_ro - r_rp * r_po) / np.sqrt((1 - r_rp**2) * (1 - r_po**2)))**2
ip = np.where(PY)[0]
ev_all = hc14.ev_rev(*A.sws[ip].T)[0]
ev_wo = loo_ev(*A.sws[ip].T)
contrib = ev_all - ev_wo
print("EV without each pair against the table's EV.minus: max abs difference", np.max(np.abs(ev_wo - ev["EV"].minus)))
print("pair order equal to MinusIndex:", (cells[ip] == ev["MinusIndex"]).all())
print("contribution range:", contrib.min(), contrib.max())

In [ ]:
# Test-run correlations: pre-run and post-run test epochs, readings (a) whole epoch, (b) laps only.
test_r = {(rd, ep): np.full(len(cells), np.nan) for rd in ("a", "b") for ep in ("prerun", "postrun")}
for rat, idx, f in SESS:
    m, h, b = session_pairs(rat, idx)
    if not len(m):
        continue
    d = D[f]
    x = hc14.position_cm(d["whl"])[:, 0]
    for ep in ("prerun", "postrun"):
        iv = hc14.intervals(d, ep)
        if not iv:
            continue
        test_r[("a", ep)][m] = pair_corr(d, m, h, b, iv)[0]
        s0 = int(iv[0][0] * hc14.WHL_FS)
        lp = [((s0 + a) / hc14.WHL_FS, (s0 + e) / hc14.WHL_FS)
              for a, e, _ in hc14.laps(x[s0:int(iv[-1][1] * hc14.WHL_FS)])]
        if lp:
            test_r[("b", ep)][m] = pair_corr(d, m, h, b, lp)[0]
d_nrem = (A.sws[:, 2] - A.sws[:, 0])[ip]
d_test = {rd: (test_r[(rd, "postrun")] - test_r[(rd, "prerun")])[ip] for rd in ("a", "b")}
order = np.argsort(-contrib)          # strongest contributors first
def fig7_test(sel_idx, rd):
    dt, dn = d_test[rd][sel_idx], d_nrem[sel_idx]
    ok = ~np.isnan(dt) & ~np.isnan(dn)
    r_, p_ = stats.pearsonr(dt[ok], dn[ok])
    return int(ok.sum()), r_, p_
n25 = int(round(0.025 * len(ip)))
nq = len(ip) // 4
groups7 = {"top 2.5%": order[:n25], "bottom 2.5%": order[-n25:], "top quartile": order[:nq],
           "second quartile": order[nq:2 * nq], "third quartile": order[2 * nq:3 * nq], "bottom quartile": order[-nq:],
           "middle 95%": order[n25:-n25]}
fig7 = {}
for rd in ("a", "b"):
    for g, sel in groups7.items():
        fig7[(rd, g)] = fig7_test(sel, rd)
print(pd.DataFrame({k: dict(zip(["pairs", "r", "p"], v)) for k, v in fig7.items()}).T.round(5).to_string())

### fig7b-strong-pairs-correlated-changes

> For the 2.5% most strongly contributing hippocampus-BLA pairs, the change in correlation from pre-run
> to post-run is positively correlated with the change from pre-NREM to post-NREM.

In [ ]:
for rd in ("a", "b"):
    n_, r_, p_ = fig7[(rd, "top 2.5%")]
    print(f"reading ({rd}): top 2.5% ({n25} pairs, {n_} defined in both test epochs): r = {r_:.3f}, p = {p_:.3g}")
    sub = order[:n25]
    rr = cells[ip][sub, 0]
    for rat in (8, 10, 11):
        s = sub[rr == rat]
        if len(s) > 3:
            nn, r2, p2 = fig7_test(s, rd)
            print(f"   rat {rat}: {nn} pairs, r = {r2:.3f}, p = {p2:.3g}")

**Result: reproduced.** Under reading (a), whole test epochs, the 867 strongest pairs defined in both test runs (the paper's 866) show r = 0.27 (p = 1.9e-16) between the test-run and NREM changes, in each rat (r = 0.26 to 0.31). Under reading (b), laps only, r = 0.084 (p = 0.028).

### fig7b-weak-pairs-not-correlated

> For the 2.5% least contributing pairs, the run and NREM correlation changes are not correlated.

In [ ]:
for rd in ("a", "b"):
    n_, r_, p_ = fig7[(rd, "bottom 2.5%")]
    print(f"reading ({rd}): bottom 2.5%: {n_} pairs, r = {r_:.3f}, p = {p_:.3g}")

**Result: reproduced.** Under reading (a), which reproduces the strong-pair claim, the 846 weakest pairs give r = 0.034 (p = 0.33); under (b), r = 0.014 (p = 0.72).

### fig7c-quartiles

> The relation holds for the most strongly contributing quartile of pairs (r = 0.13) and not for the
> least contributing quartile (r = 0.013).

In [ ]:
for rd in ("a", "b"):
    for g in ("top quartile", "second quartile", "third quartile", "bottom quartile", "middle 95%"):
        n_, r_, p_ = fig7[(rd, g)]
        print(f"reading ({rd}): {g}: {n_} pairs, r = {r_:.3f}, p = {p_:.3g}")

**Result: reproduced.** Under reading (a) the top quartile gives r = 0.130 (p = 8e-33) and the bottom quartile r = 0.014 (p = 0.20). The second quartile is weakly significant (r = 0.024, p = 0.040), the third not (p = 0.56), and the middle 95% gives r = 0.022 (p = 0.0002).

### bla-rates-safe-airpuff-absent

> Firing rates of BLA cells do not differ significantly between safe and air puff trajectories, for
> pyramidal cells or all cells.

Reading (a): per BLA cell, spikes over time on all laps of each direction in the run. Reading (b): the
mean rate within 40 cm of the air puff location on laps of each direction. Sessions: those of the
pyramidal reactivation analyses (at least 6 pyramidal cells per structure, 25) for pyramidal cells, those
with at least 6 cells of any type (27) for all cells, and every behaviour session with BLA cells as a
variant. BLA is the B1 label.

In [ ]:
ev_nrem_all = session_ev("sws", False)
sess_pyr = set(zip(ev_nrem_pyr.rat, ev_nrem_pyr.session))
sess_all = set(zip(ev_nrem_all.rat, ev_nrem_all.session))
print("sessions: pyramidal set", len(sess_pyr), "| all-cell set", len(sess_all))
rate_rows = []
for _, r in P.iterrows():
    if np.isnan(r["loc"]):
        continue
    rat, idx = r.rat, r.session
    d = D[r.folder]
    run = hc14.intervals(d, "run")
    x = hc14.position_cm(d["whl"])[:, 0]
    s0 = int(run[0][0] * hc14.WHL_FS)
    L = hc14.laps(x[s0:int(run[-1][1] * hc14.WHL_FS)])
    t = np.arange(len(x)) / hc14.WHL_FS
    near = np.abs(x - r["loc"]) <= 40
    for dirn, name in [(r["dir"], "airpuff"), (-r["dir"], "safe")]:
        ivs = [((s0 + a) / hc14.WHL_FS, (s0 + e) / hc14.WHL_FS) for a, e, dr in L if dr == dirn]
        lap_mask = np.zeros(len(x), bool)
        for a, e, dr in L:
            if dr == dirn:
                lap_mask[s0 + a:s0 + e] = True
        near_ivs_t = t[lap_mask & near]
        dur_a = sum(e - a for a, e in ivs)
        dur_b = len(near_ivs_t) / hc14.WHL_FS
        for sh, c, _ in d["units"]:
            if lab.get((rat, idx, sh)) != "BLA":
                continue
            sp = d["spikes"].get((sh, c), np.array([]))
            na = sum(np.sum((sp >= a) & (sp < e)) for a, e in ivs)
            ib = np.round(sp * hc14.WHL_FS).astype(int)
            ib = ib[(ib >= 0) & (ib < len(x))]
            nb = np.sum(lap_mask[ib] & near[ib])
            rate_rows.append(dict(rat=rat, session=idx, shank=sh, clu=c, type=ft_type.get((rat, idx, sh, c)), dir=name,
                                  a=na / dur_a if dur_a else np.nan, b=nb / dur_b if dur_b else np.nan))
RR = pd.DataFrame(rate_rows)
RW = RR.pivot_table(index=["rat", "session", "shank", "clu", "type"], columns="dir", values=["a", "b"]).reset_index()
def safe_test(df, label):
    out = []
    for rd in ("a", "b"):
        x1, x2 = df[(rd, "airpuff")], df[(rd, "safe")]
        ok = x1.notna() & x2.notna() & ((x1 + x2) > 0)
        w = stats.wilcoxon(x1[ok], x2[ok])
        out.append(f"reading ({rd}): n = {ok.sum()}, median air puff {x1[ok].median():.3f} Hz, safe {x2[ok].median():.3f} Hz, "
                   f"median difference {np.median(x1[ok] - x2[ok]):.4f} Hz, p = {w.pvalue:.3g}")
    print(label); print("  " + "\n  ".join(out))
in_pyr = [(a, b) in sess_pyr for a, b in zip(RW.rat, RW.session)]
in_all = [(a, b) in sess_all for a, b in zip(RW.rat, RW.session)]
safe_test(RW[np.array(in_pyr) & (RW.type == 1)], "pyramidal cells, 25-session set")
safe_test(RW[np.array(in_all)], "all cells, 27-session set")
safe_test(RW[RW.type == 1], "variant: pyramidal cells, all behaviour sessions")
safe_test(RW, "variant: all cells, all behaviour sessions")

**Result: reproduced.** Under reading (a), BLA rates on air puff and safe laps do not differ for pyramidal cells of the 25 sessions (median difference 0.001 Hz, p = 0.97) or for all cells of the 27 sessions (p = 0.68). Reading (b), near the air puff location, gives p = 0.060 and 0.38. Over all behaviour sessions the variant finds lower rates on air puff laps (pyramidal p = 0.013).

### fig5b-top-contributors-gain-increase

> Upmodulated BLA cells in the quartile contributing most to reactivations increase their SPW-R gain from
> pre-NREM to post-NREM, while upmodulated cells of the other three quartiles do not.

The ripple gains are those of the deposited AllRats-AllRippleCCGs table (pre-NREM and post-NREM, 10-ms
bins over +-2 s, `gain2`; the 1-ms version over +-0.2 s, `gain1`, is a variant). How the table smoothed
them is not documented. Contribution per BLA cell: the mean contribution of its pyramidal pairs; cells
split into quartiles of that mean.

In [ ]:
rcg = hc14.load_ripple_ccgs()
acc, idxpp = rcg["allccg"], rcg["idxall"].prepost.astype(int)
t2 = acc.t2
print("CCG table: cells with pre/post gains", len(idxpp), "| bins", len(t2), "from", t2[0], "to", t2[-1], "s")
pp_row = {key(r[:4]): i for i, r in enumerate(idxpp)}
bla_of_pair = [bla_key[i] for i in ip]
cdf = pd.Series(contrib).groupby(pd.Index(bla_of_pair)).mean()
q = pd.qcut(cdf, 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
print("BLA cells with pyramidal pairs:", len(cdf), "per quartile:", q.value_counts().sort_index().to_dict())
def gain_test(gname, win=0.25):
    tt = t2 if gname == "gain2" else acc.t1
    w = np.abs(tt) <= win
    pre_g, post_g = getattr(acc.pre, gname), getattr(acc.post, gname)
    rows = []
    for qq in ["Q4 highest", "Q3", "Q2", "Q1 lowest", "Q1-Q3"]:
        ks = cdf.index[(q == qq) if qq != "Q1-Q3" else (q != "Q4 highest")]
        ks = [k for k in ks if prm_key.get(k) is not None and prm_key[k][5] < 5e-4 and k in pp_row]
        a = np.array([np.nanmean(pre_g[pp_row[k], w]) for k in ks]); b = np.array([np.nanmean(post_g[pp_row[k], w]) for k in ks])
        ok = ~np.isnan(a) & ~np.isnan(b)
        wt = stats.wilcoxon(b[ok], a[ok], alternative="greater")
        rows.append(dict(quartile=qq, upmodulated=int(ok.sum()), median_pre=np.median(a[ok]), median_post=np.median(b[ok]),
                         median_post_minus_pre=np.median(b[ok] - a[ok]), p_one_sided=wt.pvalue))
    return pd.DataFrame(rows)
print("gain2 (10-ms bins), +-250 ms:"); print(gain_test("gain2").round(5).to_string())
print("variant gain1 (1-ms bins, +-200 ms):"); print(gain_test("gain1", 0.2).round(5).to_string())

**Result: reproduced.** The 63 upmodulated cells of the top quartile (the paper's 63) increase their ripple gain from pre-NREM to post-NREM (one-sided p = 5e-5), and the 74 of the other quartiles do not (p = 0.99; paper 73). The 1-ms gains give the same (p = 8e-5 and 0.98).

## Decoded sleep states

The hand-scored sleep states are not in the data set, and rescoring them from the LFP would mean streaming
461 GB of tar.gz archives (the survey above), more than the notebook can do. For the claims that
need sleep states on pairs or cells the pair table does not cover (piriform, central nuclei,
within-structure), on time within NREM (the 20-min windows, ripple rates) or on the reactivation strength,
the states are decoded from the authors' own state firing rates (`hc14.decode_states`, an assumption of
this attempt): each 5-s epoch takes the StatesFR state (columns 5 to 8) under which the spike counts of
all the session's units are most likely, with the log likelihoods summed over 60 s. NREM is column 6 and
REM column 5, in immobile epochs of pre-sleep and post-sleep. The decoder rebuilds the authors' labels
from the rates their labels produced, so it is checked below against what it does not use: the minutes
of each state implied by StatesFR, and the authors' pre and post pair correlations for NREM and REM.
The first revision's substitute (head speed below 4 cm/s over 1 s for 30 s, from unsmoothed positions)
and the decoder with 30-s smoothing are variants, run in the section "Sensitivity of the decoded states".


In [ ]:
# hc14._zbins on synthetic input: two touching 5-s intervals give all 200 bins of 50 ms.
print("bins from two touching 5-s intervals:", hc14._zbins([np.arange(0, 10, 0.013)], [(0, 5), (5, 10)])[0].shape[1], "(200 expected)")
# Minutes of NREM and REM per session: decoded, and implied by StatesFR (non-negative least squares of
# each unit's spike count over the session on its four state rates, which gives the time in each state).
from scipy.optimize import nnls
rows = []
for rat, idx, f in SESS:
    d = D[f]
    m = np.where((sfr[:, 0] == rat) & (sfr[:, 1] == idx))[0]
    n = {(sh, c): k for sh, c, k in d["units"]}
    X = sfr[m, 5:9]
    y = np.array([n.get((int(a), int(b)), np.nan) for a, b in sfr[m, 2:4]])
    ok = ~np.isnan(X).any(1) & ~np.isnan(y)
    Tfit = nnls(X[ok], y[ok])[0] / 60 if ok.sum() >= 8 else np.full(4, np.nan)
    dur = lambda k, s=("pre", "post"): sum(b - a for side in s for a, b in d[k][side]) / 60
    sleep = sum(b - a for _, nm, a, b in d["subsessions"] if nm in ("presleep", "postsleep")) / 60
    rows.append(dict(folder=f, rat=rat, sleep_min=sleep, units=len(d["states"]["units"]),
                     nrem_fit=Tfit[1], nrem=dur("nrem"), nrem_pre=dur("nrem", ("pre",)), nrem_post=dur("nrem", ("post",)),
                     nrem_30s=dur("nrem_30s"), nrem_imm=dur("nrem_imm"),
                     rem_fit=Tfit[0], rem=dur("rem"), rem_pre=dur("rem", ("pre",)), rem_post=dur("rem", ("post",)),
                     other_fit=Tfit[3]))
durs = pd.DataFrame(rows)
print(durs.round(1).to_string())
med = durs.median(numeric_only=True)
print("medians (min):", med.round(1).to_dict())
# sessions with pre- and post-sleep whose units match the table (the fit is 0 where they do not)
for a, b in [("nrem", "nrem_fit"), ("nrem_imm", "nrem_fit"), ("rem", "rem_fit")]:
    ok = durs[a].notna() & durs[b].notna() & (durs.sleep_min > 0) & (durs.nrem_fit > 0)
    print(f"{a} against {b}: median ratio {np.median(durs[a][ok] / durs[b][ok]):.2f}, Spearman rho over sessions "
          f"{stats.spearmanr(durs[a][ok], durs[b][ok])[0]:.2f} (n = {ok.sum()})")
sl = durs[durs.sleep_min > 0]
print("decoded NREM as a fraction of pre- and post-sleep, median:", np.median(sl.nrem / sl.sleep_min).round(3),
      "| first revision's substitute:", np.median(sl.nrem_imm / sl.sleep_min).round(3), "| sessions without sleep subsessions:",
      durs[durs.sleep_min == 0].folder.tolist())


In [ ]:
# Pre and post pair correlations of hippocampus-BLA pairs over the decoded states against the authors' table
# (NREM: Allcorr.sws, REM: Allcorr.rem); per-session EV - REV of pyramidal pairs, decoded against table.
def sub_corr(kind):
    pre, post = np.full(len(cells), np.nan), np.full(len(cells), np.nan)
    for rat, idx, f in SESS:
        m, h, b = session_pairs(rat, idx)
        d = D[f]
        if not len(m):
            continue
        if d[kind]["pre"]:
            pre[m] = pair_corr(d, m, h, b, d[kind]["pre"])[0]
        if d[kind]["post"]:
            post[m] = pair_corr(d, m, h, b, d[kind]["post"])[0]
    return pre, post
SUBC = {k: sub_corr(k) for k in ("nrem", "nrem_30s", "nrem_imm", "rem", "rem_30s")}
sub_pre, sub_post = SUBC["nrem"]
c_ = lambda x, y: np.corrcoef(x[~np.isnan(x) & ~np.isnan(y)], y[~np.isnan(x) & ~np.isnan(y)])[0, 1]
for kind in SUBC:
    pre, post = SUBC[kind]
    print(f"{kind}: pair correlations against the table's NREM columns pre r = {c_(pre[PY], A.sws[PY, 0]):.3f}, "
          f"post r = {c_(post[PY], A.sws[PY, 2]):.3f}; against its REM columns pre r = {c_(pre[PY], A.rem[PY, 0]):.3f}, "
          f"post r = {c_(post[PY], A.rem[PY, 2]):.3f} (pyramidal pairs)")
print(f"reference: the table's NREM against its REM columns, pre r = {c_(A.sws[PY, 0], A.rem[PY, 0]):.3f}, "
      f"post r = {c_(A.sws[PY, 2], A.rem[PY, 2]):.3f}")

def sub_session_ev(kind, col, ref):
    pre, post = SUBC[kind]
    T_ = getattr(A, col)
    rows = []
    for _, r in ref.iterrows():
        m = np.where((cells[:, 0] == r.rat) & (cells[:, 1] == r.session) & (Ty == 1).all(1))[0]
        e2, r2, n2 = hc14.ev_rev(pre[m], T_[m, 1], post[m])
        rows.append(dict(folder=r.folder, rat=r.rat, diff_table=r["diff"], diff_decoded=(e2 - r2) * 100, pairs=n2))
    return pd.DataFrame(rows)
for kind, col, ref in [("nrem", "sws", ev_nrem_pyr), ("nrem_30s", "sws", ev_nrem_pyr), ("nrem_imm", "sws", ev_nrem_pyr),
                       ("rem", "rem", ev_rem_pyr), ("rem_30s", "rem", ev_rem_pyr)]:
    e = sub_session_ev(kind, col, ref)
    ok = e.diff_decoded.notna()
    print(f"{kind} against table {col}: sessions {ok.sum()} of {len(e)}; Spearman of per-session EV - REV rho = "
          f"{stats.spearmanr(e.diff_table[ok], e.diff_decoded[ok])[0]:.3f}; mean EV - REV table {e.diff_table[ok].mean():.2f}%, "
          f"decoded {e.diff_decoded[ok].mean():.2f}%; EV > REV in {np.sum(e.diff_decoded[ok] > 0)}, signed-rank p = "
          f"{stats.wilcoxon(e.diff_decoded[ok]).pvalue:.3g}")
    if kind == "nrem":
        print(e.round(3).to_string())


### fig3c-decay (decoded NREM)

> Hippocampus-BLA reactivation decays over the first hour of NREM sleep (mean EV - REV of 2.89, 1.72 and
> 1.31% in successive 20-min epochs), while staying significant in each epoch.

The windows are the first, second and third 20 min of decoded post-NREM time (when post-NREM is shorter
than 40 min, the second window takes what remains and the third is undefined); pre-NREM (decoded) and run
whole; pyramidal pairs of the pair table; sessions with at least 6 cells per structure and EV > REV in the
first window. A variant takes pre-NREM from the table.


In [ ]:
def cut_windows(ivs, width=1200.0, n=3):
    """Split a list of intervals into consecutive windows of `width` s of cumulative time."""
    out, cur, acc = [[] for _ in range(n)], 0, 0.0
    for a, b in ivs:
        while a < b and cur < n:
            take = min(b - a, width - acc)
            out[cur].append((a, a + take)); acc += take; a += take
            if acc >= width - 1e-9:
                cur, acc = cur + 1, 0.0
    total = sum(b - a for a, b in ivs)
    if total < 2 * width:          # the second window takes what remains, the third is undefined
        out[2] = []
    return out

def windows_ev(pyr=True, pre_src="decoded"):
    rows = []
    for rat, idx, f in SESS:
        d = D[f]
        m = (cells[:, 0] == rat) & (cells[:, 1] == idx)
        if pyr:
            m &= (Ty == 1).all(1)
        m = np.where(m)[0]
        if not len(m) or not d["nrem"]["pre"] or not d["nrem"]["post"]:
            continue
        h = [tuple(r) for r in cells[m][:, 2:4]]; b = [tuple(r) for r in cells[m][:, 4:6]]
        if len(set(h)) < 6 or len(set(b)) < 6:
            continue
        pre = sub_pre[m] if pre_src == "decoded" else A.sws[m, 0]
        row = dict(rat=rat, session=idx, folder=f, post_nrem_min=sum(y - x for x, y in d["nrem"]["post"]) / 60)
        for w, iv in enumerate(cut_windows(d["nrem"]["post"])):
            if not iv:
                row[f"w{w+1}"] = np.nan; continue
            post, _ = pair_corr(d, m, h, b, iv)
            e, r_, n_ = hc14.ev_rev(pre, A.sws[m, 1], post)
            row[f"w{w+1}"] = (e - r_) * 100
            row[f"pairs{w+1}"] = n_
            for side, ks in (("hpc", set(h)), ("bla", set(b))):
                row[f"{side}_spikes{w+1}"] = sum(hc14.in_intervals(d["spikes"].get(k, np.array([])), iv).sum() for k in ks)
        rows.append(row)
    return pd.DataFrame(rows)

def decay_report(W, label):
    sel = W[W.w1 > 0]
    print(f"{label}: qualifying sessions {len(W)}, with EV > REV in the first window {len(sel)} "
          f"(per rat {sel.groupby('rat').size().to_dict()})")
    gap = sel[sel.w2.isna() & sel.w3.notna()]
    if len(gap):    # a window inside the post-NREM time with no pair defined
        print("  sessions with window 2 undefined and window 3 defined (pairs defined, spikes of the "
              "hippocampal and BLA cells per window):")
        print(gap[["folder", "post_nrem_min"] + [f"{c}{w}" for c in ("pairs", "hpc_spikes", "bla_spikes") for w in (1, 2, 3)]].to_string())
    for w in ("w1", "w2", "w3"):
        x = sel[w].dropna()
        p = stats.wilcoxon(x).pvalue if len(x) > 1 else np.nan
        print(f"  {w}: n = {len(x)}, mean EV-REV {x.mean():.2f} +- {x.sem():.2f}%, signed-rank p = {p:.3g}")
    groups = [sel[w].dropna() for w in ("w1", "w2", "w3")]
    kw = stats.kruskal(*groups)
    print(f"  Kruskal-Wallis across windows: chi2 = {kw.statistic:.2f}, p = {kw.pvalue:.3g}")
    return sel, kw
W_pyr = windows_ev(True)
decay_pyr = decay_report(W_pyr, "pyramidal pairs")
# Variant: pre-NREM correlations from the authors' table, post-NREM windows decoded.
decay_report(windows_ev(True, "table"), "variant, table pre-NREM, pyramidal pairs")


**Result: reproduced (resting on the decoded NREM).** Among 17 sessions with EV > REV in the first window (the paper's 19; the claims file's tolerance is 1.9, so the count is just outside it), the mean EV - REV is 3.04 +- 0.55, 1.78 +- 0.53 and 1.69 +- 0.88% in the three 20-min windows, each within the paper's s.e.m. of 2.89, 1.72 and 1.31%; EV exceeds REV in each window (p = 1.5e-5, 0.00021, 0.035) and the decay test is significant (Kruskal-Wallis p = 0.024). With the table's pre-NREM, 20 sessions, the means are 2.51, 1.52 and 1.30% and the decay test holds (p = 0.021), but the third window gives p = 0.067 (partially reproduced). In that variant Rat10-20140622 has window 2 undefined: its hippocampal cells fire no spike in that window (0 spikes, against 20,844 and 19,394 in windows 1 and 3) while its BLA cells fire 31,040, so no pair is defined there. The main reading lists no such session among its 17. The variants in the sensitivity section give partially reproduced with 30-s smoothing (first mean 3.44%, outside the s.e.m.) and not reproduced with the first revision's substitute.


### figs5a-hpc-bla-all-cells

> Hippocampus-BLA reactivation in NREM sleep holds when interneurons are included, is absent in REM sleep,
> and decays over the first hour of NREM.

NREM and REM from the pair table (all cell types); the 20-min windows from the decoded NREM, as in
fig3c-decay.

In [ ]:
ev_test(session_ev("sws", False), "NREM, all pairs (table)")
ev_rem_all = session_ev("rem", False)
ev_test(ev_rem_all, "REM, all pairs (table)")
ev_test(ev_rem_all[[(a, b) in rem_ok for a, b in zip(ev_rem_all.rat, ev_rem_all.session)]],
        "variant, REM, all pairs, more than 180 s of decoded REM")
W_all = windows_ev(False)
decay_all = decay_report(W_all, "all pairs, decoded windows")
decay_report(windows_ev(False, "table"), "variant, table pre-NREM, all pairs")

**Result: reproduced (the windows rest on the decoded NREM).** With all cell types, NREM EV exceeds REV over 27 sessions (p = 3.5e-5, the paper's 27) and REM does not (p = 0.37). In the decoded windows the first two 20-min windows are significant (p = 1.9e-6 and 0.011, 20 sessions); the third is not (p = 0.47). With the table's pre-NREM the first two give p = 3.8e-6 and 0.00016. With the 180-s requirement on decoded REM, REM gives p = 0.24 over 22 sessions.


### fig3d-pir-weaker-than-bla (decoded NREM)

> Piriform cortex neurons reactivate less with their CA1 partners than BLA neurons do (mean EV - REV 0.85%
> against 3.13%).

Hippocampus-Pir and hippocampus-BLA pyramidal pairs, with the decoded NREM for both and run correlations
from the raw spikes; at least 6 pyramidal cells per structure.


In [ ]:
def struct_cells(d, rat, idx, f, which, pyr=True):
    out = []
    for sh, c, _ in d["units"]:
        if which == "hpc":
            ok = sh in hpc_shanks.get(rat, set())
        else:
            ok = lab.get((rat, idx, sh)) == which
        if ok and (not pyr or ft_type.get((rat, idx, sh, c)) == 1) and (sh, c) in d["spikes"]:
            out.append((sh, c))
    return out

def cross_ev(which, pyr=True, state="nrem", min_s=0.0):
    rows = []
    for rat, idx, f in SESS:
        d = D[f]
        if not d[state]["pre"] or not d[state]["post"] or not hc14.intervals(d, "run"):
            continue
        if min(sum(b - a for a, b in d[state][s]) for s in ("pre", "post")) <= min_s:
            continue
        H, S2 = struct_cells(d, rat, idx, f, "hpc", pyr), struct_cells(d, rat, idx, f, which, pyr)
        if len(H) < 6 or len(S2) < 6:
            continue
        run = hc14.intervals(d, "run")
        cs = []
        for ivs in (d[state]["pre"], run, d[state]["post"]):
            a = np.hstack([hc14.bin_counts([d["spikes"][k] for k in H], x, y) for x, y in ivs])
            b = np.hstack([hc14.bin_counts([d["spikes"][k] for k in S2], x, y) for x, y in ivs])
            cs.append(hc14.cross_corr(a, b).ravel())
        e, r_, n = hc14.ev_rev(*cs)
        rows.append(dict(rat=rat, session=idx, folder=f, n_hpc=len(H), n_other=len(S2), pairs=n, diff=(e - r_) * 100,
                         EV=e, REV=r_))
    return pd.DataFrame(rows)
pir_sub = cross_ev("Pir")
bla_sub = cross_ev("BLA")
for name, df in [("hippocampus-Pir", pir_sub), ("hippocampus-BLA", bla_sub)]:
    p = stats.wilcoxon(df.EV, df.REV).pvalue
    print(f"{name}: sessions {len(df)} (per rat {df.groupby('rat').size().to_dict()}), mean EV-REV {df['diff'].mean():.2f} "
          f"+- {df['diff'].sem():.2f}%, EV > REV in {np.sum(df['diff'] > 0)}, signed-rank p = {p:.3g}")
both_ = pir_sub[pir_sub.folder.isin(bla_sub.folder)]
for lab_, pdf in [("all Pir sessions", pir_sub), ("Pir sessions also qualifying for BLA", both_)]:
    rs = stats.mannwhitneyu(bla_sub["diff"], pdf["diff"], alternative="greater")
    z = stats.norm.isf(rs.pvalue)
    print(f"{lab_} (n = {len(pdf)}, mean {pdf['diff'].mean():.2f}%): one-tailed rank-sum BLA > Pir p = {rs.pvalue:.3g} (z = {z:.2f})")
for name, df in [("hippocampus-Pir", pir_sub), ("hippocampus-BLA", bla_sub)]:
    print(f"{name} per rat, mean EV-REV:", df.groupby("rat")["diff"].agg(["size", "mean"]).round(2).to_dict("index"))

**Result: reproduced (resting on the decoded NREM).** Over 14 hippocampus-Pir sessions (7, 6 and 1 in rats 8, 10 and 11; the paper's 14) the mean EV - REV is 0.87 +- 0.35%, against 3.09 +- 0.76% for 24 hippocampus-BLA sessions, both within the paper's s.e.m., and the one-tailed rank-sum test gives p = 0.027. On the 10 Pir sessions that also qualify for BLA, the mean is 1.03% and p = 0.063.


### figs4-intra-structure-nrem (decoded NREM)

> Within the hippocampus and within the BLA, cell pairs reactivate during NREM sleep, with all cells and
> with pyramidal cells only.

In [ ]:
def intra_ev(which, pyr, state="nrem", min_s=0.0):
    rows = []
    for rat, idx, f in SESS:
        d = D[f]
        if not d[state]["pre"] or not d[state]["post"] or not hc14.intervals(d, "run"):
            continue
        if min(sum(b - a for a, b in d[state][s]) for s in ("pre", "post")) <= min_s:
            continue
        K = struct_cells(d, rat, idx, f, which, pyr)
        if len(K) < 6:
            continue
        run = hc14.intervals(d, "run")
        iu = np.triu_indices(len(K), 1)
        cs = []
        for ivs in (d[state]["pre"], run, d[state]["post"]):
            a = np.hstack([hc14.bin_counts([d["spikes"][k] for k in K], x, y) for x, y in ivs])
            cs.append(hc14.cross_corr(a, a)[iu])
        e, r_, n = hc14.ev_rev(*cs)
        rows.append(dict(rat=rat, session=idx, cells=len(K), pairs=n, EV=e, REV=r_))
    return pd.DataFrame(rows)
intra = {}
for which in ("hpc", "BLA"):
    for pyr in (False, True):
        df = intra_ev(which, pyr)
        intra[(which, pyr)] = df
        w = stats.wilcoxon(df.EV, df.REV)
        print(f"{which}, {'pyramidal' if pyr else 'all'} cells: sessions {len(df)} (per rat {df.groupby('rat').size().to_dict()}), "
              f"median EV {df.EV.median()*100:.2f}%, REV {df.REV.median()*100:.2f}%, EV > REV in {np.sum(df.EV > df.REV)}, p = {w.pvalue:.3g}")

**Result: reproduced (resting on the decoded NREM).** Within the hippocampus (all cells, 38 sessions, p = 6.5e-7; pyramidal, 38 sessions, p = 2.5e-6) and within the BLA (all cells, 36 sessions, p = 0.0031; pyramidal, 34 sessions, p = 0.00047), EV exceeds REV. The session counts are close to the paper's 38, 38, 34 and 32.


### ripple-rate-pre-post-absent (decoded NREM)

> The occurrence rate of SPW-Rs does not differ between pre-NREM and post-NREM.

In [ ]:
rows = []
for rat, idx, f in SESS:
    d = D[f]
    rp = d["ripple_peaks"]
    if not len(rp) or not d["nrem"]["pre"] or not d["nrem"]["post"]:
        continue
    r0 = hc14.in_intervals(rp, d["nrem"]["pre"]).sum() / sum(b - a for a, b in d["nrem"]["pre"])
    r1 = hc14.in_intervals(rp, d["nrem"]["post"]).sum() / sum(b - a for a, b in d["nrem"]["post"])
    rows.append(dict(rat=rat, folder=f, pre=r0, post=r1))
RRt = pd.DataFrame(rows)
w2 = stats.wilcoxon(RRt.pre, RRt.post)
w1 = stats.wilcoxon(RRt.post, RRt.pre, alternative="greater")
print(f"sessions {len(RRt)} (per rat {RRt.groupby('rat').size().to_dict()}); median rate pre {RRt.pre.median():.3f}/s, "
      f"post {RRt.post.median():.3f}/s, median post-pre {np.median(RRt.post - RRt.pre):.4f}/s; two-sided p = {w2.pvalue:.3g}; "
      f"variant one-sided (post > pre) p = {w1.pvalue:.3g}")

**Result: reproduced (resting on the decoded NREM).** Over 39 sessions the ripple rate in pre-NREM (0.506/s) and post-NREM (0.516/s) does not differ (two-sided p = 0.80; one-sided p = 0.61).


## Reactivation strength around ripples (decoded NREM)

Templates from the run (whole run, air puff laps, safe laps), cells of the pair table in each qualifying
session (at least 6 cells of the type in each structure), ripple peaks of rip.evt that fall in the
decoded pre-NREM and post-NREM; per session the mean R over +-250 ms.

In [ ]:
def rs_session(rat, idx, f, pyr):
    d = D[f]
    m = (cells[:, 0] == rat) & (cells[:, 1] == idx)
    hk = sorted({tuple(r) for r in cells[m][:, 2:4]}); bk = sorted({tuple(r) for r in cells[m][:, 4:6]})
    if pyr:
        hk = [k for k in hk if ft_type.get((rat, idx) + k) == 1]; bk = [k for k in bk if ft_type.get((rat, idx) + k) == 1]
    if len(hk) < 6 or len(bk) < 6 or not d["nrem"]["pre"] or not d["nrem"]["post"]:
        return None
    sh, sb = [d["spikes"][k] for k in hk], [d["spikes"][k] for k in bk]
    run = hc14.intervals(d, "run")
    x = hc14.position_cm(d["whl"])[:, 0]
    s0 = int(run[0][0] * hc14.WHL_FS)
    L = hc14.laps(x[s0:int(run[-1][1] * hc14.WHL_FS)])
    dirn = P.set_index("folder").loc[f, "dir"]
    tmpl = {"whole": run,
            "airpuff": [((s0 + a) / hc14.WHL_FS, (s0 + e) / hc14.WHL_FS) for a, e, dr in L if dr == dirn],
            "safe": [((s0 + a) / hc14.WHL_FS, (s0 + e) / hc14.WHL_FS) for a, e, dr in L if dr != dirn]}
    out = dict(rat=rat, session=idx, folder=f, n_hpc=len(hk), n_bla=len(bk))
    sleep = {side: hc14.sleep_bins(sh, sb, d["nrem"][side]) for side in ("pre", "post")}
    for name, iv in tmpl.items():
        for side in ("pre", "post"):
            lags, prof, n = hc14.reactivation_strength(sh, sb, iv, sleep[side], d["ripple_peaks"])
            w = np.abs(lags) <= 0.25 + 1e-9
            out[f"{name}_{side}"] = np.nanmean(prof[w]); out[f"{name}_{side}_peak"] = prof[len(prof) // 2]
            out[f"n_rip_{side}"] = n
    return out
RS = {}
for pyr in (True, False):
    RS[pyr] = pd.DataFrame([r for r in (rs_session(rat, idx, f, pyr) for rat, idx, f in SESS
                                        if (rat, idx) in rip_sessions) if r is not None])
    print("pyramidal" if pyr else "all cells", "sessions:", len(RS[pyr]), "per rat", RS[pyr].groupby("rat").size().to_dict(),
          "| ripples used, median pre/post:", RS[pyr].n_rip_pre.median(), RS[pyr].n_rip_post.median())
def rs_test(df, name, label):
    a, b = df[f"{name}_post"], df[f"{name}_pre"]
    w = stats.wilcoxon(a, b, alternative="greater")
    print(f"{label}: n = {len(df)}, median pre {b.median():.3f}, post {a.median():.3f}, median post-pre {np.median(a - b):.3f}, "
          f"post > pre in {np.sum(a > b)}, one-sided p = {w.pvalue:.3g}")
    return w.pvalue

### fig6a-airpuff-trajectory-reactivation

> Reactivation strength of the air puff trajectory's hippocampus-BLA correlation pattern around SPW-Rs is
> higher in post-NREM than in pre-NREM.

In [ ]:
rs_test(RS[True], "airpuff", "air puff template, pyramidal cells")

**Result: reproduced (resting on the decoded NREM).** Over 24 sessions the air puff template's peri-ripple strength is higher in post-NREM (20 of 24, one-sided p = 0.00037).


### fig6a-safe-trajectory-no-reactivation

> Reactivation strength of the safe trajectory's pattern around SPW-Rs does not differ significantly
> between post-NREM and pre-NREM.

In [ ]:
rs_test(RS[True], "safe", "safe template, pyramidal cells")

**Result: reproduced (resting on the decoded NREM).** The safe template's strength does not rise significantly (15 of 24 higher in post-NREM, median difference 0.002, p = 0.060). The margin is small: with 30-s smoothing and with the first revision's substitute, p = 0.048 (not reproduced).


### fig6b-airpuff-greater-than-safe

> The post-NREM minus pre-NREM difference in peri-ripple reactivation strength is larger for the air puff
> trajectory than for the safe trajectory.

In [ ]:
def diff_test(df, label, suffix=""):
    da = df[f"airpuff_post{suffix}"] - df[f"airpuff_pre{suffix}"]
    ds = df[f"safe_post{suffix}"] - df[f"safe_pre{suffix}"]
    w = stats.wilcoxon(da, ds, alternative="greater")
    print(f"{label}: n = {len(df)}, median difference air puff {da.median():.3f}, safe {ds.median():.3f}, "
          f"median (air puff - safe) {np.median(da - ds):.3f}, one-sided p = {w.pvalue:.3g}")
    return w.pvalue
diff_test(RS[True], "pyramidal cells, mean over +-250 ms")
diff_test(RS[True], "variant, peak bin", "_peak")

**Result: reproduced (resting on the decoded NREM).** The post minus pre difference is larger for the air puff template (median 0.010 against 0.002; one-sided p = 0.035; peak bin p = 0.015). With 30-s smoothing p = 0.045; with the first revision's substitute p = 0.15.


### figs11a-whole-run-reactivation

> Reactivation strength of the whole training session's pattern around SPW-Rs is higher in post-NREM than
> in pre-NREM, for pyramidal cell pairs and for all cell pairs.

In [ ]:
rs_test(RS[True], "whole", "whole-run template, pyramidal cells")
rs_test(RS[False], "whole", "whole-run template, all cells")

**Result: reproduced (resting on the decoded NREM).** The whole-run template's strength is higher in post-NREM for pyramidal cells (21 of 24, p = 4.5e-5) and all cells (18 of 26, p = 0.00072).


### figs11b-all-cells-both-trajectories

> When interneurons are included, reactivation strength rises from pre-NREM to post-NREM for both the air
> puff and the safe trajectory.

In [ ]:
rs_test(RS[False], "airpuff", "air puff template, all cells")
rs_test(RS[False], "safe", "safe template, all cells")

**Result: reproduced (resting on the decoded NREM).** With all cells, both templates rise from pre-NREM to post-NREM (air puff p = 0.011, safe p = 0.0018, 26 sessions).


### figs11c-all-cells-difference-absent

> When interneurons are included, the post minus pre reactivation difference does not differ
> significantly between the safe and air puff trajectories.

In [ ]:
diff_test(RS[False], "all cells, mean over +-250 ms")
diff_test(RS[False], "variant, peak bin", "_peak")

**Result: reproduced (resting on the decoded NREM).** With all cells the air puff and safe differences do not differ (median difference 0.004, p = 0.38; peak bin p = 0.31).


## Claims that need REM (decoded REM)

fig3d-pir-reactivation, fig3e-cen-no-reactivation and figs4-intra-structure-rem-absent need REM for pairs
the authors' table does not cover. They are tested on the decoded REM (StatesFR column 5, immobile
epochs of pre- and post-sleep), checked above against the table's REM pair correlations. A session needs
more than 180 s of REM in pre-sleep and in post-sleep (claims file, opening comment) and at least 6 cells
of the type used in each structure.

### fig3d-pir-reactivation

> Hippocampus-piriform cortex pyramidal pairs reactivate during NREM sleep but not REM sleep.


In [ ]:
def rem_report(df, name):
    n0 = len(df)
    df = df.dropna(subset=["EV", "REV"]) if len(df) else df
    if n0 > len(df):
        print(f"{name}: {n0 - len(df)} session(s) with EV undefined left out")
    if len(df) > 1:
        w = stats.wilcoxon(df.EV, df.REV)
        d_ = (df.EV - df.REV) * 100
        print(f"{name}: sessions {len(df)} (per rat {df.groupby('rat').size().to_dict()}), median EV-REV {d_.median():.2f}%, "
              f"EV > REV in {np.sum(d_ > 0)}, signed-rank p = {w.pvalue:.3g}")
        return w.pvalue
    print(f"{name}: sessions {len(df)}")
    return np.nan
pir_rem = cross_ev("Pir", True, "rem", 180)
p_pir = rem_report(pir_sub, "hippocampus-Pir, pyramidal, NREM"), rem_report(pir_rem, "hippocampus-Pir, pyramidal, REM")


**Result: reproduced (resting on the decoded NREM and REM).** Hippocampus-Pir pyramidal pairs reactivate in NREM (14 sessions, EV > REV in 9, p = 0.042) and not in REM (11 sessions with more than 180 s of REM, p = 0.18). The NREM margin is small, and with 30-s smoothing REM gives p = 0.032 (EV > REV in 1 of 11, so REV is the larger), which would make the claim partially reproduced.


### fig3e-cen-no-reactivation

> There are no reactivations between the central nuclei and the hippocampus, in NREM or REM sleep.


In [ ]:
cen_all = cross_ev("CeCM", pyr=False)
cen_rem = cross_ev("CeCM", False, "rem", 180)
p_cen = rem_report(cen_all, "hippocampus-CeN, all cells, NREM"), rem_report(cen_rem, "hippocampus-CeN, all cells, REM")


**Result: reproduced (resting on the decoded NREM and REM).** Hippocampus-CeN pairs of all cell types show no reactivation in NREM (18 sessions, median EV - REV 0.22%, EV > REV in 12, p = 0.73) or in REM (16 sessions, median -0.05%, EV > REV in 6, p = 0.90).


### figs4-intra-structure-rem-absent

> Within the hippocampus and within the BLA, there is no significant reactivation during REM sleep.


In [ ]:
intra_rem = {}
for which in ("hpc", "BLA"):
    for pyr in (False, True):
        intra_rem[(which, pyr)] = intra_ev(which, pyr, "rem", 180)
        rem_report(intra_rem[(which, pyr)], f"{which}, {'pyramidal' if pyr else 'all'} cells, REM")


**Result: reproduced (resting on the decoded REM).** No REM reactivation within the hippocampus (all cells, 31 sessions, p = 0.31, one session with EV undefined left out; pyramidal, 31 sessions, p = 1.0) or within the BLA (all cells, 28 sessions, p = 0.39; pyramidal, 26 sessions, p = 0.47). With 30-s smoothing all four also give p >= 0.05 (0.37, 0.87, 0.23, 0.20).


## Sensitivity of the decoded states

The claims that rest on the decoded states are re-run with two variants: the decoder with 30-s smoothing
(`nrem_30s`, `rem_30s`), and the first revision's substitute NREM (`nrem_imm`, head speed below 4 cm/s over
1 s for at least 30 s, no REM). The 2-cm/s threshold of the first
revision is replaced by these.


In [ ]:
keep = {f: (D[f]["nrem"], D[f]["rem"]) for _, _, f in SESS}
sub_pre_main = sub_pre
for var_n, var_r in [("nrem_30s", "rem_30s"), ("nrem_imm", None)]:
    print(f"===== variant {var_n}" + (f" / {var_r}" if var_r else ""))
    for _, _, f in SESS:
        D[f]["nrem"] = D[f][var_n]
        if var_r:
            D[f]["rem"] = D[f][var_r]
    sub_pre = SUBC[var_n][0]
    decay_report(windows_ev(True), "fig3c-decay, pyramidal pairs")
    decay_report(windows_ev(False), "figs5a windows, all pairs")
    RSv = {pyr: pd.DataFrame([r for r in (rs_session(rat, idx, f, pyr) for rat, idx, f in SESS if (rat, idx) in rip_sessions)
                              if r is not None]) for pyr in (True, False)}
    rs_test(RSv[True], "airpuff", "fig6a air puff"); rs_test(RSv[True], "safe", "fig6a safe")
    diff_test(RSv[True], "fig6b"); rs_test(RSv[True], "whole", "figs11a pyramidal")
    rs_test(RSv[False], "whole", "figs11a all cells"); rs_test(RSv[False], "airpuff", "figs11b air puff")
    rs_test(RSv[False], "safe", "figs11b safe"); diff_test(RSv[False], "figs11c")
    rows = []
    for rat, idx, f in SESS:
        d = D[f]; rp = d["ripple_peaks"]
        if len(rp) and d["nrem"]["pre"] and d["nrem"]["post"]:
            rows.append((hc14.in_intervals(rp, d["nrem"]["pre"]).sum() / sum(b - a for a, b in d["nrem"]["pre"]),
                         hc14.in_intervals(rp, d["nrem"]["post"]).sum() / sum(b - a for a, b in d["nrem"]["post"])))
    rows = np.array(rows)
    print(f"ripple rate: sessions {len(rows)}, median pre {np.median(rows[:, 0]):.3f}, post {np.median(rows[:, 1]):.3f}, "
          f"two-sided p = {stats.wilcoxon(rows[:, 0], rows[:, 1]).pvalue:.3g}")
    pir_v, bla_v = cross_ev("Pir"), cross_ev("BLA")
    print(f"fig3d-pir-weaker: Pir {len(pir_v)} sessions mean {pir_v['diff'].mean():.2f}%, BLA {len(bla_v)} sessions mean "
          f"{bla_v['diff'].mean():.2f}%, one-tailed rank-sum p = {stats.mannwhitneyu(bla_v['diff'], pir_v['diff'], alternative='greater').pvalue:.3g}")
    for which in ("hpc", "BLA"):
        for pyr in (False, True):
            df = intra_ev(which, pyr)
            print(f"figs4-intra NREM {which}, {'pyramidal' if pyr else 'all'}: sessions {len(df)}, p = {stats.wilcoxon(df.EV, df.REV).pvalue:.3g}")
    rem_report(pir_v, "fig3d-pir NREM")
    rem_report(cross_ev("CeCM", pyr=False), "fig3e CeN NREM")
    if var_r:
        rem_report(cross_ev("Pir", True, "rem", 180), "fig3d-pir REM")
        rem_report(cross_ev("CeCM", False, "rem", 180), "fig3e CeN REM")
        for which in ("hpc", "BLA"):
            for pyr in (False, True):
                rem_report(intra_ev(which, pyr, "rem", 180), f"figs4-intra REM {which}, {'pyramidal' if pyr else 'all'}")
for f, (a, b) in keep.items():
    D[f]["nrem"], D[f]["rem"] = a, b
sub_pre = sub_pre_main


In [ ]:
print(f"notebook run time up to here: {(time.time() - T0) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| units-recorded | reproduced |
| fig1b-dz-speed-anova | partially reproduced (assumption: air puff location) |
| fig1b-dz-speed-posthoc | partially reproduced (assumption: air puff location) |
| fig1b-speed-ratio | reproduced (assumption: air puff location) |
| figs1b-ratio-each-rat | reproduced (assumption: air puff location) |
| fig1c-slower-in-dz-post-run | reproduced (assumption: air puff location) |
| figs1c-habituation | reproduced |
| airpuff-location-no-bias | not reproduced (assumption: air puff location) |
| fig2c-bla-pyr-higher-in-sleep | reproduced (assumption: REM column) |
| fig2c-bla-interneurons | reproduced (assumption: REM column) |
| fig2c-skewed-rates | reproduced |
| fig3c-hpc-bla-ev-nrem | reproduced |
| fig3c-hpc-bla-ev-rem-absent | reproduced |
| fig3c-decay | reproduced (assumption: decoded NREM) |
| figs5a-hpc-bla-all-cells | reproduced (assumption: decoded NREM) |
| fig3d-pir-reactivation | reproduced (assumption: decoded NREM and REM) |
| fig3d-pir-weaker-than-bla | reproduced (assumption: decoded NREM) |
| fig3e-cen-no-reactivation | reproduced (assumption: decoded NREM and REM) |
| figs4-intra-structure-nrem | reproduced (assumption: decoded NREM) |
| figs4-intra-structure-rem-absent | reproduced (assumption: decoded REM) |
| fig4-bla-ripple-modulation | reproduced |
| figs5b-ripple-modulated-pairs-reactivate-more | reproduced |
| run-correlated-pairs | reproduced |
| fig5a-up-positive-pairs-change-most | reproduced |
| figs6b-ev-nine-groups | reproduced |
| fig5b-top-contributors-gain-increase | reproduced |
| figs9-ev-independent-of-rate | reproduced |
| fig6a-airpuff-trajectory-reactivation | reproduced (assumption: decoded NREM) |
| fig6a-safe-trajectory-no-reactivation | reproduced (assumption: decoded NREM) |
| fig6b-airpuff-greater-than-safe | reproduced (assumption: decoded NREM) |
| figs11a-whole-run-reactivation | reproduced (assumption: decoded NREM) |
| figs11b-all-cells-both-trajectories | reproduced (assumption: decoded NREM) |
| figs11c-all-cells-difference-absent | reproduced (assumption: decoded NREM) |
| bla-rates-safe-airpuff-absent | reproduced |
| ripple-rate-pre-post-absent | reproduced (assumption: decoded NREM) |
| fig7b-strong-pairs-correlated-changes | reproduced |
| fig7b-weak-pairs-not-correlated | reproduced |
| fig7c-quartiles | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

How the analysis differs from the paper's. The hand-scored sleep states are not deposited, so the NREM
and REM claims on hippocampus-BLA pairs, cell rates and ripple modulation use the authors' own summary
tables, and the other sleep claims use states decoded from the authors' per-state firing rates (StatesFR)
in immobile epochs of pre- and post-sleep. The decoded NREM and REM give hippocampus-BLA pair
correlations close to the authors' (r = 0.88 and 0.90 for NREM, 0.78 and 0.81 for REM) and per-session
NREM EV - REV close to theirs (mean 3.13% against 3.21%, Spearman 0.92), but they are not the authors'
intervals. Several outcomes on them sit near p = 0.05 and change with the decoder's smoothing or with the
first revision's immobility substitute: fig3c-decay, fig3d-pir-reactivation, fig3d-pir-weaker-than-bla,
fig6a-safe-trajectory and fig6b. The air puff location of each session is estimated from the head position at the air puffs; with the location corrected by the six airpuffloc files and all samples, fig1b-speed-ratio would be
partially reproduced and fig1b-dz-speed-posthoc and figs1b-ratio-each-rat not reproduced. Laps follow the claims file's description of
ClassifyLaps.m, not the code itself.

What was found about the dataset. The summary tables (AllRats-Variables) carry no documentation; their
columns were identified here (StatesFR column 5 as REM is an assumption, supported by the decoded REM)
and they agree with the raw spikes where that can be checked: the units of 60 of 61 sessions, the session
rates, and the run correlations of the pair table (r > 0.999 in 17 of 29 sessions, 0.58 to 0.98 in the
other 12). They reproduce the paper's numbers closely, and the paper's per-pair results follow from them.
The sleep scoring is not deposited, nor the run epochs used for the pair table. Rat08-20130722 has 25
more raw units than the tables. The structure lists come in four formats, one per rat; Rat08's are in a
spreadsheet whose "All" sheet holds the labels as text. Only 6 of 55 behaviour sessions have an air puff
location file, and there the head position at the air puffs lies 7.3 cm (median) past the file's
location in the direction of travel. In Rat10-20140622 the hippocampal cells of the pair table fire no
spike over a 20-min stretch of decoded post-sleep NREM in which its BLA cells fire 31,040 spikes.
